# Score-Based Generative Modeling through Stochastic Differential Equations

## Noise Conditional Score Network (NCSN++)

Version 1.3

Source:

   https://github.com/yang-song/score_sde_pytorch/ 

Adapted by:

   Antonio Esteves @ UMinho, April 2025

In [ ]:
import os
import ml_collections
import time
import math
import string
import abc
import wandb
import glob
import random
from   shutil                       import rmtree, copy
import numpy                        as     np
from   scipy                        import integrate
from   pathlib                      import Path
import PIL.Image                    as     Image
import matplotlib.pyplot            as     plt
from   tqdm.notebook                import tqdm
from   natsort                      import natsorted
from   functools                    import partial
import torch
import torch.nn                     as     nn
import torch.nn.functional          as     F
import torch.optim                  as     optim
from   torchvision.utils            import save_image, make_grid
from   torch.autograd               import Function
from   torch.utils.cpp_extension    import load
import torchvision.datasets         as     dsets
from   torch.utils.data             import DataLoader, Dataset
from   torchvision                  import transforms
from   torchinfo                    import summary
import torch_fidelity


## Configuration

In [ ]:
def get_celeba_config():
  '''
  Defines the configuration for training with CelebA dataset and NCSN++ model.
  '''
  config = ml_collections.ConfigDict()

  config.training   = training   = ml_collections.ConfigDict()
  config.sampling   = sampling   = ml_collections.ConfigDict()
  config.evaluate   = evaluate   = ml_collections.ConfigDict()
  config.data       = data       = ml_collections.ConfigDict()
  config.model      = model      = ml_collections.ConfigDict()
  config.optim      = optim      = ml_collections.ConfigDict()
  config.experiment = experiment = ml_collections.ConfigDict()

  experiment.experiment_name     = "NCSNpp_07"
  experiment.root_dir            = "OUR_WORK_DIR"
  experiment.results_dir         = "results"
  experiment.models_dir          = "models"
  # UNCOMMENT FOR TRAINING ........................................................
  experiment.mode                = "train"   # "train" or "evaluate" or "summary"
  experiment.saved_model         = None      # Path to saved model to be restored or 'None' to start from scratch
  # UNCOMMENT FOR EVALUATION ......................................................
  # experiment.mode              = "evaluate"   # "train" or "evaluate" or "summary"
  # experiment.saved_model       = "OUR_WORK_DIR/models/NCSNpp_07/NCSNpp_07_epochXYZ_stepABC.pth"      # Path to saved model to be restored or 'None' to start from scratch
                    
  # training .....................................................

  training.accumulated_batch_size  = 64   # Accumulated batch size (used for gradient accumulation).
  training.batch_size              = 8    # Effective batch size
  training.batches_accum_gradients = int(training.accumulated_batch_size // training.batch_size) # Number of batches to accumulate gradents.
  training.epochs               = 100     # Number of epochs
  training.n_iters              = None    # Number of iterations.
                                          # It is calculated based on the number of epochs and dataset size.
  training.snapshot_freq        = 1       # Interval between saving successive model checkpoints (in epochs). 
                                          # Later it is updated to iterations.
  training.log_freq             = 100     # Interval between creating successive log entries (in iterations).
  training.eval_freq            = 1000    # Interval between successive model evaluations (in iterations).
  training.sde                  = "vesde" # The type of Stochastic Differential Equation adopted ('vesde', 'vpsde', 'subvpsde').
  training.continuous           = False   # Continuous (True) or discrete (False) SDE.
  training.snapshot_sampling    = True    # Produce samples during training (True) or not (False).
  training.likelihood_weighting = False   # Weight the mixture of score matching losses according to 
                                          # https://arxiv.org/abs/2101.09258 (True) or use the weighting 
                                          # scheme proposed in the NCSN paper (False). 
  training.reduce_mean          = False   # Average (True) or sum (False) the loss across data dimensions.

  # sampling .....................................................

  sampling.n_steps_each         = 1          # The number of corrector steps per predictor update.
  sampling.noise_removal        = True       # Add one-step denoising to the final samples (True) or not (False).
  sampling.probability_flow     = False      # Create the reverse-time ODE used in probability flow ODE sampling (True) or not (False).
  sampling.snr                  = 0.17       # The signal-to-noise ratio for configuring correctors.
  sampling.method               = "pc"       # Probability flow ODE sampling ('ode') or predictor-corrector sampling ('pc').
  sampling.predictor            = "reverse_diffusion" # Predictor method: 'euler_maruyama', 'reverse_diffusion', or 'ancestral_sampling'.
  sampling.corrector            = "langevin" # Corrector method: 'langevin' (Langevin dynamics) or 'ald' (annealed Langevin dynamics).

  # evaluation ....................................................

  evaluate.data_path               = "OUR_DATASETS_DIR/celeba/balanced_split/val" # Path to the evaluation dataset.
  evaluate.batch_size              = 8     # Batch size during evaluation.
  evaluate.enable_sampling         = True  # Produce samples during evaluation (True) or not (False).
  evaluate.num_samples             = 1600  # 50000 # Number of samples to use during evaluation.
  evaluate.enable_loss             = True  # Compute the loss on the full evaluation dataset (True) or not (False).

  # UNCOMMENT TO CALCULATE THE NEGATIVE LOG-LIKELIHOOD ........................................
  # evaluate.enable_bpd            = True  # Estimate the log-likelihood (True) or not (False).

  # UNCOMMENT TO DO NOT CALCULATE THE NEGATIVE LOG-LIKELIHOOD .................................
  evaluate.enable_bpd              = False # Estimate the log-likelihood (True) or not (False).

  evaluate.bpd_dataset             = "test" # Use the training images ('train') or evaluation images ('test') during evaluation.
  evaluate.metrics_dir             = 'metrics' # Directory to save the evaluation metrics.
  evaluate.feature_layer_fid       = '192'  # Identifier of the features' layer used to compute FID: '64', '192', '768', '2048', 'logits_unbiased', 'logits'.
  evaluate.kid_subset_size         = 128    # Number of samples in each subset for computing KID.
  evaluate.samples_resize_crop     = 128    # Resize/crop images to this size. 0 means do not resize/crop images.
  evaluate.clear_metrics_cache     = True   # Clear the cache of computed metrics (True) or not (False).
  evaluate.calc_metrics_interval   = 0  #5000  # Interval between successive calculations of metrics. (int) 0 means do not compute metrics during training.
  evaluate.calc_metrics_batch_size = 16     # Batch size for computing metrics.
  evaluate.calc_metrics_path       = "OUR_DATASETS_DIR/celeba/balanced_split/train" # Path to the real images used to compute metrics.

  # data ..........................................................

  data.dataset                  = "CELEBA balanced"                            # Dataset name.
  data.data_path                = "OUR_DATASETS_DIR/celeba/balanced_split/train" # Path to the training dataset.
  data.image_size               = 128   # Image size.
  data.random_flip              = True  # Apply random flip transformation to the images during training (True) or not (False).
  data.uniform_dequantization   = False # Add uniform dequantization to the images (True) or not (False).
  data.centered                 = False # Scale the images values to the range [-1, 1] (True) or keep them at [0, 1] (False).
  data.num_channels             = 3     # Number of channels in the images.
  data.crop_size                = 128   # Crop the images to this size.
  data.resize                   = True  # Resize the images to the specified size (True) or not (False).
  data.normalize                = False # Normalize the images to have zero mean and unit variance (True) or not (False).
  data.centercrop               = False # Crop the central part of the images (True) or not (False).

  # model ..........................................................

  model.name                    = "ncsnpp" # Model name.
  model.scale_by_sigma          = True     # Scale the NCSN++ output by the added noise standard deviation (True) or not (False).
  model.sigma_begin             = 90
  model.sigma_max               = 90.0     # Maximum noise variance.
  model.sigma_min               = 0.01     # Minimum noise variance.
  model.num_scales              = 1000     # Number of noise levels applied.
  model.beta_min                = 0.1      # Minimum noise variance applied in a diffusion step when using DDPM model.
  model.beta_max                = 20.0     # Maximum noise variance applied in a diffusion step when using DDPM model.
  model.dropout                 = 0.1      # Percentage of elements that are zeroed out in Dropout layers.
  model.ema_rate                = 0.999    # The decay rate applied when computing the exponential moving averages of the model parameters.
  model.normalization           = "GroupNorm" # The type of normalization layer to use: 'InstanceNorm', 'InstanceNorm++', 'VarianceNorm' or 'GroupNorm'.
  model.nonlinearity            = "swish"  # The type of activation function to use: 'elu', 'relu', 'lrelu' (LeakyReLU), 'swish' (SiLU).
  model.nf                      = 128      # Timestep embedding dimension.
  model.ch_mult                 = (1, 2, 2, 2) # Multiplier factor applied to 'model.nf' ir order to obtain the number of output channels in NCSN++ downsampling blocks.
  model.num_res_blocks          = 4         # Number of ResNet blocks (for each resolution) in the NCSN++ downsampling part. 
  model.attn_resolutions        = (16,)     # Image resolutions at which to apply self-attention.
  model.resamp_with_conv        = True      # Include a convolution layer in the 'Upsample' block.
  model.conditional             = True      # Noise-level conditioned model (True) or not (False).
  model.fir                     = True      # Use a finite impulse response filter (FIR) in the 'Upsample' and 'Downsample' blocks.
  model.fir_kernel              = [1, 3, 3, 1] # FIR kernel.
  model.skip_rescale            = True       # Skip the rescaling operation at the end of the attention blocks (True) or not (False).
  model.resblock_type           = "ddpm"     # ResNet block type: 'biggan' or 'ddpm'.
  model.progressive             = "none"     # Use progressive training: 'none', 'output_skip', 'residual'.
  model.progressive_input       = "residual" # Progressive input type: 'residual' or 'input_skip'.
  model.progressive_combine     = "sum"      # Progressive combine type: 'sum' or 'cat' (concatenate).
  model.attention_type          = "ddpm"     # Attention type.
  model.init_scale              = 0.0        # Weights and biases initialization scale.
  model.conv_size               = 3          # Convolution kernel size (not used in NCSn++).
  model.embedding_type          = "positional" # Type of embedding used with timestep: 'fourier' (Gaussian Fourier features embedding)) 
                                               # or 'positional' (sinusoidal positional embedding).

  # optimization ...................................................

  optim.weight_decay            = 0      # Weight decay.
  optim.optimizer               = "Adam" # The optimizer.
  optim.lr                      = 0.0002 # Learning rate.
  optim.beta1                   = 0.9    # Beta1 parameter of the Adam optimizer.
  optim.eps                     = 1e-8   # Epsilon parameter of the Adam optimizer.
  optim.warmup                  = 5000   # Number of warmup steps where the learning rate is smaller than 'lr'.
  optim.grad_clip               = 1.0    # Maximum values for the gradients norm.

  # other parameters ...............................................

  config.score_name             = "ScoreModel"
  config.seed                   = 42

  if experiment.mode == "summary":
    config.device = 'cpu'
  else:
    config.device = torch.device('cuda') if torch.cuda.is_available() else torch.device('cpu')

  return config

def print_config(config):
  '''
  Prints the configuration.
  '''
  print('Configuration parameters:')
  for name, values in config.items():
      if isinstance(values, ml_collections.config_dict.config_dict.ConfigDict):
          print(f'{name}:')
          for key, value in values.items():
              print(f'\t{key}: {value}')
      else:
          print(f'{name}: {values}')

## Setup the environment

Read the configuration and create the necessary folders.

In [ ]:
config = get_celeba_config()
print(f'Using {config.device} for computing')

# Print training configuration .................................................

print_config(config)

# Location where we will save here the images generated during model training
RESULTS_PATH = os.path.join(
    config.experiment.root_dir,
    config.experiment.results_dir,
    config.experiment.experiment_name
)
os.makedirs(RESULTS_PATH, exist_ok=True)

# Location where the trained models will be saved
MODELS_PATH  = os.path.join(
    config.experiment.root_dir,
    config.experiment.models_dir,
    config.experiment.experiment_name
)
os.makedirs(MODELS_PATH, exist_ok=True)

## Utilities

In [ ]:
def to_flattened_numpy(x):
    """
    Flatten a torch tensor `x` and convert it to numpy.
    """
    return x.detach().cpu().numpy().reshape((-1,))


def from_flattened_numpy(x, shape):
    """
    Form a torch tensor with the given `shape` from a flattened numpy array `x`.
    """
    return torch.from_numpy(x.reshape(shape))

def time_format(seconds: int) -> str:
    '''
    Converts a time in seconds to days:hours:minutes:seconds.
    '''
    if seconds is not None:
        seconds = int(seconds)
        d = seconds // (3600 * 24)
        h = seconds // 3600 % 24
        m = seconds % 3600 // 60
        s = seconds % 3600 % 60
        if d > 0:
            return '{:02d}D {:02d}H {:02d}m {:02d}s'.format(d, h, m, s)
        elif h > 0:
            return '{:02d}H {:02d}m {:02d}s'.format(h, m, s)
        elif m > 0:
            return '{:02d}m {:02d}s'.format(m, s)
        elif s > 0:
            return '{:02d}s'.format(s)
    return '-'


## Normalization Layers

In [ ]:
def get_normalization(config, conditional=False):
  """
  Returns the normalization layer specified in the configuration file.
  """
  norm = config.model.normalization
  if conditional:
    if norm == 'InstanceNorm++':
      return partial(ConditionalInstanceNorm2dPlus, num_classes=config.model.num_classes)
    else:
      raise NotImplementedError(f'{norm} not implemented yet.')
  else:
    if norm == 'InstanceNorm':
      return nn.InstanceNorm2d
    elif norm == 'InstanceNorm++':
      return InstanceNorm2dPlus
    elif norm == 'VarianceNorm':
      return VarianceNorm2d
    elif norm == 'GroupNorm':
      return nn.GroupNorm
    else:
      raise ValueError('Unknown normalization: %s' % norm)


class ConditionalBatchNorm2d(nn.Module):
  def __init__(self, num_features, num_classes, bias=True):
    super().__init__()
    self.num_features = num_features
    self.bias         = bias
    self.bn           = nn.BatchNorm2d(num_features, affine=False)
    if self.bias:
      self.embed = nn.Embedding(num_classes, num_features * 2)
      self.embed.weight.data[:, :num_features].uniform_()  # Initialize scale at N(1, 0.02)
      self.embed.weight.data[:, num_features:].zero_()     # Initialize bias at 0
    else:
      self.embed = nn.Embedding(num_classes, num_features)
      self.embed.weight.data.uniform_()

  def forward(self, x, y):
    out = self.bn(x)
    if self.bias:
      gamma, beta = self.embed(y).chunk(2, dim=1)
      out         = gamma.view(-1, self.num_features, 1, 1) * out \
                    + beta.view(-1, self.num_features, 1, 1)
    else:
      gamma = self.embed(y)
      out   = gamma.view(-1, self.num_features, 1, 1) * out
    return out


class ConditionalInstanceNorm2d(nn.Module):
  def __init__(self, num_features, num_classes, bias=True):
    super().__init__()
    self.num_features  = num_features
    self.bias          = bias
    self.instance_norm = nn.InstanceNorm2d(
      num_features,
      affine=False,
      track_running_stats=False
    )
    if bias:
      self.embed = nn.Embedding(num_classes, num_features * 2)
      self.embed.weight.data[:, :num_features].uniform_()  # Initialize scale at N(1, 0.02)
      self.embed.weight.data[:, num_features:].zero_()     # Initialize bias at 0
    else:
      self.embed = nn.Embedding(num_classes, num_features)
      self.embed.weight.data.uniform_()

  def forward(self, x, y):
    h = self.instance_norm(x)
    if self.bias:
      gamma, beta = self.embed(y).chunk(2, dim=-1)
      out         = gamma.view(-1, self.num_features, 1, 1) * h \
                    + beta.view(-1, self.num_features, 1, 1)
    else:
      gamma = self.embed(y)
      out   = gamma.view(-1, self.num_features, 1, 1) * h
    return out


class ConditionalVarianceNorm2d(nn.Module):
  def __init__(self, num_features, num_classes, bias=False):
    super().__init__()
    self.num_features = num_features
    self.bias         = bias
    self.embed        = nn.Embedding(num_classes, num_features)
    self.embed.weight.data.normal_(1, 0.02)

  def forward(self, x, y):
    vars  = torch.var(x, dim=(2, 3), keepdim=True)
    h     = x / torch.sqrt(vars + 1e-5)

    gamma = self.embed(y)
    out   = gamma.view(-1, self.num_features, 1, 1) * h
    return out


class VarianceNorm2d(nn.Module):
  def __init__(self, num_features, bias=False):
    super().__init__()
    self.num_features = num_features
    self.bias         = bias
    self.alpha        = nn.Parameter(torch.zeros(num_features))
    self.alpha.data.normal_(1, 0.02)

  def forward(self, x):
    vars = torch.var(x, dim=(2, 3), keepdim=True)
    h    = x / torch.sqrt(vars + 1e-5)

    out  = self.alpha.view(-1, self.num_features, 1, 1) * h
    return out


class ConditionalNoneNorm2d(nn.Module):
  def __init__(self, num_features, num_classes, bias=True):
    super().__init__()
    self.num_features = num_features
    self.bias         = bias
    if bias:
      self.embed = nn.Embedding(num_classes, num_features * 2)
      self.embed.weight.data[:, :num_features].uniform_()  # Initialize scale at N(1, 0.02)
      self.embed.weight.data[:, num_features:].zero_()     # Initialize bias at 0
    else:
      self.embed = nn.Embedding(num_classes, num_features)
      self.embed.weight.data.uniform_()

  def forward(self, x, y):
    if self.bias:
      gamma, beta = self.embed(y).chunk(2, dim=-1)
      out         = gamma.view(-1, self.num_features, 1, 1) * x \
                    + beta.view(-1, self.num_features, 1, 1)
    else:
      gamma = self.embed(y)
      out   = gamma.view(-1, self.num_features, 1, 1) * x
    return out


class NoneNorm2d(nn.Module):
  def __init__(self, num_features, bias=True):
    super().__init__()

  def forward(self, x):
    return x


class InstanceNorm2dPlus(nn.Module):
  def __init__(self, num_features, bias=True):
    super().__init__()
    self.num_features  = num_features
    self.bias          = bias
    self.instance_norm = nn.InstanceNorm2d(num_features, affine=False, track_running_stats=False)
    self.alpha         = nn.Parameter(torch.zeros(num_features))
    self.gamma         = nn.Parameter(torch.zeros(num_features))
    self.alpha.data.normal_(1, 0.02)
    self.gamma.data.normal_(1, 0.02)
    if bias:
      self.beta = nn.Parameter(torch.zeros(num_features))

  def forward(self, x):
    means = torch.mean(x, dim=(2, 3))
    m     = torch.mean(means, dim=-1, keepdim=True)
    v     = torch.var(means, dim=-1, keepdim=True)
    means = (means - m) / (torch.sqrt(v + 1e-5))
    h     = self.instance_norm(x)

    if self.bias:
      h   = h + means[..., None, None] * self.alpha[..., None, None]
      out = self.gamma.view(-1, self.num_features, 1, 1) * h + self.beta.view(-1, self.num_features, 1, 1)
    else:
      h   = h + means[..., None, None] * self.alpha[..., None, None]
      out = self.gamma.view(-1, self.num_features, 1, 1) * h
    return out


class ConditionalInstanceNorm2dPlus(nn.Module):
  def __init__(self, num_features, num_classes, bias=True):
    super().__init__()
    self.num_features  = num_features
    self.bias          = bias
    self.instance_norm = nn.InstanceNorm2d(num_features, affine=False, track_running_stats=False)
    if bias:
      self.embed = nn.Embedding(num_classes, num_features * 3)
      self.embed.weight.data[:, :2 * num_features].normal_(1, 0.02)  # Initialize scale at N(1, 0.02)
      self.embed.weight.data[:, 2 * num_features:].zero_()           # Initialize bias at 0
    else:
      self.embed = nn.Embedding(num_classes, 2 * num_features)
      self.embed.weight.data.normal_(1, 0.02)

  def forward(self, x, y):
    means = torch.mean(x, dim=(2, 3))
    m     = torch.mean(means, dim=-1, keepdim=True)
    v     = torch.var(means, dim=-1, keepdim=True)
    means = (means - m) / (torch.sqrt(v + 1e-5))
    h     = self.instance_norm(x)

    if self.bias:
      gamma, alpha, beta = self.embed(y).chunk(3, dim=-1)
      h   = h + means[..., None, None] * alpha[..., None, None]
      out = gamma.view(-1, self.num_features, 1, 1) * h + beta.view(-1, self.num_features, 1, 1)
    else:
      gamma, alpha = self.embed(y).chunk(2, dim=-1)
      h   = h + means[..., None, None] * alpha[..., None, None]
      out = gamma.view(-1, self.num_features, 1, 1) * h
    return out


## Layers used for up-sampling or down-sampling images 

In [ ]:
module_path  = os.getcwd()
upfirdn2d_op = load(
    "upfirdn2d",
    sources=[
        os.path.join(module_path, "op/upfirdn2d.cpp"),
        os.path.join(module_path, "op/upfirdn2d_kernel.cu"),
    ],
)

class UpFirDn2dBackward(Function):
    @staticmethod
    def forward(
        ctx, grad_output, kernel, grad_kernel, up, down, pad, g_pad, in_size, out_size
    ):

        up_x, up_y     = up
        down_x, down_y = down
        g_pad_x0, g_pad_x1, g_pad_y0, g_pad_y1 = g_pad

        grad_output = grad_output.reshape(-1, out_size[0], out_size[1], 1)

        grad_input = upfirdn2d_op.upfirdn2d(
            grad_output,
            grad_kernel,
            down_x,
            down_y,
            up_x,
            up_y,
            g_pad_x0,
            g_pad_x1,
            g_pad_y0,
            g_pad_y1,
        )
        grad_input = grad_input.view(in_size[0], in_size[1], in_size[2], in_size[3])

        ctx.save_for_backward(kernel)

        pad_x0, pad_x1, pad_y0, pad_y1 = pad

        ctx.up_x     = up_x
        ctx.up_y     = up_y
        ctx.down_x   = down_x
        ctx.down_y   = down_y
        ctx.pad_x0   = pad_x0
        ctx.pad_x1   = pad_x1
        ctx.pad_y0   = pad_y0
        ctx.pad_y1   = pad_y1
        ctx.in_size  = in_size
        ctx.out_size = out_size

        return grad_input

    @staticmethod
    def backward(ctx, gradgrad_input):
        kernel,        = ctx.saved_tensors

        gradgrad_input = gradgrad_input.reshape(-1, ctx.in_size[2], ctx.in_size[3], 1)

        gradgrad_out   = upfirdn2d_op.upfirdn2d(
            gradgrad_input,
            kernel,
            ctx.up_x,
            ctx.up_y,
            ctx.down_x,
            ctx.down_y,
            ctx.pad_x0,
            ctx.pad_x1,
            ctx.pad_y0,
            ctx.pad_y1,
        )
        # gradgrad_out = gradgrad_out.view(ctx.in_size[0], ctx.out_size[0], ctx.out_size[1], ctx.in_size[3])
        gradgrad_out = gradgrad_out.view(
            ctx.in_size[0], ctx.in_size[1], ctx.out_size[0], ctx.out_size[1]
        )

        return gradgrad_out, None, None, None, None, None, None, None, None


class UpFirDn2d(Function):
    @staticmethod
    def forward(ctx, input, kernel, up, down, pad):
        up_x, up_y                     = up
        down_x, down_y                 = down
        pad_x0, pad_x1, pad_y0, pad_y1 = pad

        kernel_h, kernel_w         = kernel.shape
        batch, channel, in_h, in_w = input.shape
        ctx.in_size                = input.shape

        input = input.reshape(-1, in_h, in_w, 1)

        ctx.save_for_backward(kernel, torch.flip(kernel, [0, 1]))

        out_h        = (in_h * up_y + pad_y0 + pad_y1 - kernel_h) // down_y + 1
        out_w        = (in_w * up_x + pad_x0 + pad_x1 - kernel_w) // down_x + 1
        ctx.out_size = (out_h, out_w)

        ctx.up   = (up_x, up_y)
        ctx.down = (down_x, down_y)
        ctx.pad  = (pad_x0, pad_x1, pad_y0, pad_y1)

        g_pad_x0 = kernel_w - pad_x0 - 1
        g_pad_y0 = kernel_h - pad_y0 - 1
        g_pad_x1 = in_w * up_x - out_w * down_x + pad_x0 - up_x + 1
        g_pad_y1 = in_h * up_y - out_h * down_y + pad_y0 - up_y + 1

        ctx.g_pad = (g_pad_x0, g_pad_x1, g_pad_y0, g_pad_y1)

        out = upfirdn2d_op.upfirdn2d(
            input, kernel, up_x, up_y, down_x, down_y, pad_x0, pad_x1, pad_y0, pad_y1
        )
        # out = out.view(major, out_h, out_w, minor)
        out = out.view(-1, channel, out_h, out_w)

        return out

    @staticmethod
    def backward(ctx, grad_output):
        kernel, grad_kernel = ctx.saved_tensors

        grad_input = UpFirDn2dBackward.apply(
            grad_output,
            kernel,
            grad_kernel,
            ctx.up,
            ctx.down,
            ctx.pad,
            ctx.g_pad,
            ctx.in_size,
            ctx.out_size,
        )

        return grad_input, None, None, None, None

def upfirdn2d(input, kernel, up=1, down=1, pad=(0, 0)):
    if input.device.type == "cpu":
        out = upfirdn2d_native(
            input, kernel, up, up, down, down, pad[0], pad[1], pad[0], pad[1]
        )

    else:
        out = UpFirDn2d.apply(
            input, kernel, (up, up), (down, down), (pad[0], pad[1], pad[0], pad[1])
        )

    return out


def upfirdn2d_native(
    input, kernel, up_x, up_y, down_x, down_y, pad_x0, pad_x1, pad_y0, pad_y1
    ):
    _, channel, in_h, in_w = input.shape
    input                  = input.reshape(-1, in_h, in_w, 1)

    _, in_h, in_w, minor   = input.shape
    kernel_h, kernel_w     = kernel.shape

    out = input.view(-1, in_h, 1, in_w, 1, minor)
    out = F.pad(out, [0, 0, 0, up_x - 1, 0, 0, 0, up_y - 1])
    out = out.view(-1, in_h * up_y, in_w * up_x, minor)

    out = F.pad(
        out, [0, 0, max(pad_x0, 0), max(pad_x1, 0), max(pad_y0, 0), max(pad_y1, 0)]
    )
    out = out[
        :,
        max(-pad_y0, 0) : out.shape[1] - max(-pad_y1, 0),
        max(-pad_x0, 0) : out.shape[2] - max(-pad_x1, 0),
        :,
    ]

    out = out.permute(0, 3, 1, 2)
    out = out.reshape(
        [-1, 1, in_h * up_y + pad_y0 + pad_y1, in_w * up_x + pad_x0 + pad_x1]
    )
    w   = torch.flip(kernel, [0, 1]).view(1, 1, kernel_h, kernel_w)
    out = F.conv2d(out, w)
    out = out.reshape(
        -1,
        minor,
        in_h * up_y + pad_y0 + pad_y1 - kernel_h + 1,
        in_w * up_x + pad_x0 + pad_x1 - kernel_w + 1,
    )
    out   = out.permute(0, 2, 3, 1)
    out   = out[:, ::down_y, ::down_x, :]

    out_h = (in_h * up_y + pad_y0 + pad_y1 - kernel_h) // down_y + 1
    out_w = (in_w * up_x + pad_x0 + pad_x1 - kernel_w) // down_x + 1

    return out.view(-1, channel, out_h, out_w)


In [ ]:
# torch.compiler.allow_in_graph(upfirdn2d_op.upfirdn2d)

In [ ]:
# Function ported from StyleGAN2
def get_weight(
    module,
    shape,
    weight_var  = 'weight',
    kernel_init = None,
    ):
    """
    Get/create weight tensor for a convolution or fully-connected layer.
    """
    return module.param(weight_var, kernel_init, shape)


class Conv2d(nn.Module):
    """
    Conv2d layer with optimal upsampling and downsampling (inspired by StyleGAN2).
    """

    def __init__(
        self,
        in_ch,
        out_ch,
        kernel,
        up              = False,
        down            = False,
        resample_kernel = (1, 3, 3, 1),
        use_bias        = True,
        kernel_init     = None
        ):
        super().__init__()
        assert not (up and down)
        assert kernel >= 1 and kernel % 2 == 1
        self.weight = nn.Parameter(torch.zeros(out_ch, in_ch, kernel, kernel))
        if kernel_init is not None:
            self.weight.data = kernel_init(self.weight.data.shape)
        if use_bias:
            self.bias = nn.Parameter(torch.zeros(out_ch))

        self.up              = up
        self.down            = down
        self.resample_kernel = resample_kernel
        self.kernel          = kernel
        self.use_bias        = use_bias

    def forward(self, x):
        if self.up:
            x = upsample_conv_2d(x, self.weight, k=self.resample_kernel)
        elif self.down:
            x = conv_downsample_2d(x, self.weight, k=self.resample_kernel)
        else:
            x = F.conv2d(x, self.weight, stride=1, padding=self.kernel // 2)

        if self.use_bias:
            x = x + self.bias.reshape(1, -1, 1, 1)

        return x


def naive_upsample_2d(x, factor=2):
    _N, C, H, W = x.shape
    x           = torch.reshape(x, (-1, C, H, 1, W, 1))
    x           = x.repeat(1, 1, 1, factor, 1, factor)
    return torch.reshape(x, (-1, C, H * factor, W * factor))


def naive_downsample_2d(x, factor=2):
    _N, C, H, W = x.shape
    x           = torch.reshape(x, (-1, C, H // factor, factor, W // factor, factor))
    return torch.mean(x, dim=(3, 5))


def upsample_conv_2d(x, w, k=None, factor=2, gain=1):
    """
    Fused `upsample_2d` followed by `conv2d`.

    Padding is performed only once at the beginning, not between the operations.
    The fused op is considerably more efficient than performing the same calculation
    using standard TensorFlow ops. It supports gradients of arbitrary order.

    Arguments:
    x:            Input tensor of the shape `[N, C, H, W]` or `[N, H, W, C]`.
    w:            Weight tensor of the shape `[filterH, filterW, inChannels, outChannels]`.
                  Grouped convolution can be performed by `inChannels = x.shape[0] // numGroups`.
    k:            FIR filter of the shape `[firH, firW]` or `[firN]` (separable). 
                  The default is `[1] * factor`, which corresponds to nearest-neighbor upsampling.
    factor:       Integer upsampling factor (default: 2).
    gain:         Scaling factor for signal magnitude (default: 1.0).

    Returns:
       Tensor of the shape `[N, C, H * factor, W * factor]` or
       `[N, H * factor, W * factor, C]`, and same datatype as `x`.
    """

    assert isinstance(factor, int) and factor >= 1

    # Check weight shape
    assert len(w.shape) == 4
    convH = w.shape[2]
    convW = w.shape[3]
    inC   = w.shape[1]
    outC  = w.shape[0]

    assert convW == convH

    # Setup filter kernel
    if k is None:
        k = [1] * factor
    k = _setup_kernel(k) * (gain * (factor ** 2))
    p = (k.shape[0] - factor) - (convW - 1)

    stride = (factor, factor)

    # Determine data dimensions
    stride         = [1, 1, factor, factor]
    output_shape   = ((_shape(x, 2) - 1) * factor + convH, (_shape(x, 3) - 1) * factor + convW)
    output_padding = (output_shape[0] - (_shape(x, 2) - 1) * factor - convH,
                     output_shape[1] - (_shape(x, 3) - 1) * factor - convW)
    assert output_padding[0] >= 0 and output_padding[1] >= 0
    num_groups     = _shape(x, 1) // inC

    # Transpose weights
    w = torch.reshape(w, (num_groups, -1, inC, convH, convW))

    #w = w[..., ::-1, ::-1].permute(0, 2, 1, 3, 4)
    w2 = torch.flip(w,[3,4])
    w  = w2.permute(0, 2, 1, 3, 4)

    w = torch.reshape(w, (num_groups * inC, -1, convH, convW))

    # print(f'[DEBUG] upsample_conv_2d: x shape is {x.shape}')
    # print(f'[DEBUG] upsample_conv_2d: w shape is {w.shape}')
    # print(f'[DEBUG] upsample_conv_2d: output_shape is {output_shape}')
    # print(f'[DEBUG] upsample_conv_2d: output_padding is {output_padding}')
    # print(f'[DEBUG] upsample_conv_2d: stride is {stride}')

    x = F.conv_transpose2d(x, w, stride=(factor, factor), output_padding=output_padding, padding=0)

    return upfirdn2d(
        x,
        torch.tensor(k, device=x.device),
        pad = ((p + 1) // 2 + factor - 1, p // 2 + 1)
        )


def conv_downsample_2d(x, w, k=None, factor=2, gain=1):
    """
    Fused `conv2d` followed by `downsample_2d`.

    Padding is performed only once at the beginning, not between the operations.
    The fused op is considerably more efficient than performing the same calculation
    using standard TensorFlow ops. It supports gradients of arbitrary order.
    Args:
        x:            Input tensor of the shape `[N, C, H, W]` or `[N, H, W, C]`.
        w:            Weight tensor of the shape `[filterH, filterW, inChannels, outChannels]`.
                      Grouped convolution can be performed by `inChannels = x.shape[0] // numGroups`.
        k:            FIR filter of the shape `[firH, firW]` or `[firN]` (separable).
                      The default is `[1] * factor`, which corresponds to average pooling.
        factor:       Integer downsampling factor (default: 2).
        gain:         Scaling factor for signal magnitude (default: 1.0).

    Returns:
        Tensor of the shape `[N, C, H // factor, W // factor]` or
        `[N, H // factor, W // factor, C]`, and same datatype as `x`.
    """

    assert isinstance(factor, int) and factor >= 1
    _outC, _inC, convH, convW = w.shape
    assert convW == convH
    if k is None:
        k = [1] * factor
    k = _setup_kernel(k) * gain
    p = (k.shape[0] - factor) + (convW - 1)
    s = [factor, factor]
    x = upfirdn2d(
        x,
        torch.tensor(k, device=x.device),
        pad=((p + 1) // 2, p // 2)
    )
    return F.conv2d(x, w, stride=s, padding=0)


def _setup_kernel(k):
    k = np.asarray(k, dtype=np.float32)
    if k.ndim == 1:
        k = np.outer(k, k)
    k /= np.sum(k)
    assert k.ndim == 2
    assert k.shape[0] == k.shape[1]
    return k


def _shape(x, dim):
    return x.shape[dim]


def upsample_2d(x, k=None, factor=2, gain=1):
    r"""
    Upsample a batch of 2D images with the given filter.

    Accepts a batch of 2D images of the shape `[N, C, H, W]` or `[N, H, W, C]`
    and upsamples each image with the given filter. The filter is normalized so that
    if the input pixels are constant, they will be scaled by the specified `gain`.
    Pixels outside the image are assumed to be zero, and the filter is padded with
    zeros so that its shape is a multiple of the upsampling factor.

    Args:
        x:            Input tensor of the shape `[N, C, H, W]` or `[N, H, W, C]`.
        k:            FIR filter of the shape `[firH, firW]` or `[firN]` (separable).
                      The default is `[1] * factor`, which corresponds to 
                      nearest-neighbor upsampling.
        factor:       Integer upsampling factor (default: 2).
        gain:         Scaling factor for signal magnitude (default: 1.0).

    Returns:
        Tensor of the shape `[N, C, H * factor, W * factor]`
    """
    assert isinstance(factor, int) and factor >= 1
    if k is None:
        k = [1] * factor
    k = _setup_kernel(k) * (gain * (factor ** 2))
    p = k.shape[0] - factor
    return upfirdn2d(
        x,
        torch.tensor(k, device=x.device),
        up=factor,
        pad=((p + 1) // 2 + factor - 1, p // 2)
        )


def downsample_2d(x, k=None, factor=2, gain=1):
    r"""
    Downsample a batch of 2D images with the given filter.

    Accepts a batch of 2D images of the shape `[N, C, H, W]` or `[N, H, W, C]`
    and downsamples each image with the given filter. The filter is normalized so that
    if the input pixels are constant, they will be scaled by the specified `gain`.
    Pixels outside the image are assumed to be zero, and the filter is padded with
    zeros so that its shape is a multiple of the downsampling factor.

    Args:
        x:            Input tensor of the shape `[N, C, H, W]` or `[N, H, W, C]`.
        k:            FIR filter of the shape `[firH, firW]` or `[firN]` (separable).
                      The default is `[1] * factor`, which corresponds to average pooling.
        factor:       Integer downsampling factor (default: 2).
        gain:         Scaling factor for signal magnitude (default: 1.0).

    Returns:
        Tensor of the shape `[N, C, H // factor, W // factor]`
    """

    assert isinstance(factor, int) and factor >= 1
    if k is None:
        k = [1] * factor
    k = _setup_kernel(k) * gain
    p = k.shape[0] - factor
    return upfirdn2d(
        x,
        torch.tensor(k, device=x.device),
        down=factor, 
        pad=((p + 1) // 2, p // 2)
        )


## Layers

In [ ]:
def get_act(config):
  """
  Returns the activation function selected by the configuration file.
  """
  if config.model.nonlinearity.lower() == 'elu':
    return nn.ELU()
  elif config.model.nonlinearity.lower() == 'relu':
    return nn.ReLU()
  elif config.model.nonlinearity.lower() == 'lrelu':
    return nn.LeakyReLU(negative_slope=0.2)
  elif config.model.nonlinearity.lower() == 'swish':
    return nn.SiLU()
  else:
    raise NotImplementedError('activation function does not exist!')


def ncsn_conv1x1(in_planes, out_planes, stride=1, bias=True, dilation=1, init_scale=1., padding=0):
  """
  1x1 convolution. Same as NCSNv1/v2.
  """
  conv = nn.Conv2d(
    in_planes,
    out_planes,
    kernel_size = 1,
    stride      = stride,
    bias        = bias,
    dilation    = dilation,
    padding     = padding
  )
  init_scale        = 1e-10 if init_scale == 0 else init_scale
  conv.weight.data *= init_scale
  conv.bias.data   *= init_scale
  return conv


def variance_scaling(
    scale,mode,
    distribution,
    in_axis  = 1,
    out_axis = 0,
    dtype    = torch.float32,
    device   = 'cpu',
  ):
  """
  Ported from JAX.
  """
  def _compute_fans(shape, in_axis=1, out_axis=0):
    receptive_field_size = np.prod(shape) / shape[in_axis] / shape[out_axis]
    fan_in  = shape[in_axis] * receptive_field_size
    fan_out = shape[out_axis] * receptive_field_size
    return fan_in, fan_out

  def init(shape, dtype=dtype, device=device):
    fan_in, fan_out = _compute_fans(shape, in_axis, out_axis)
    if mode == "fan_in":
      denominator = fan_in
    elif mode == "fan_out":
      denominator = fan_out
    elif mode == "fan_avg":
      denominator = (fan_in + fan_out) / 2
    else:
      raise ValueError(
        "invalid mode for variance scaling initializer: {}".format(mode))
    variance = scale / denominator
    if distribution == "normal":
      return torch.randn(*shape, dtype=dtype, device=device) * np.sqrt(variance)
    elif distribution == "uniform":
      return (torch.rand(*shape, dtype=dtype, device=device) * 2. - 1.) * np.sqrt(3 * variance)
    else:
      raise ValueError("invalid distribution for variance scaling initializer")

  return init


def default_init(scale=1.):
  """
  The same initialization used in DDPM.
  """
  scale = 1e-10 if scale == 0 else scale
  return variance_scaling(scale, 'fan_avg', 'uniform')


class Dense(nn.Module):
  """
  Linear layer with `default_init`.
  """
  def __init__(self):
    super().__init__()


def ddpm_conv1x1(in_planes, out_planes, stride=1, bias=True, init_scale=1., padding=0):
  """
  1x1 convolution with DDPM initialization.
  """
  conv = nn.Conv2d(
    in_planes,
    out_planes,
    kernel_size = 1,
    stride      = stride,
    padding     = padding,
    bias        = bias,
  )
  conv.weight.data = default_init(init_scale)(conv.weight.data.shape)
  nn.init.zeros_(conv.bias)
  return conv


def ncsn_conv3x3(in_planes, out_planes, stride=1, bias=True, dilation=1, init_scale=1., padding=1):
  """
  3x3 convolution with PyTorch initialization. Same as NCSNv1/NCSNv2.
  """
  init_scale = 1e-10 if init_scale == 0 else init_scale
  conv       = nn.Conv2d(
    in_planes,
    out_planes,
    stride      = stride,
    bias        = bias,
    dilation    = dilation,
    padding     = padding,
    kernel_size = 3,
  )
  conv.weight.data *= init_scale
  conv.bias.data   *= init_scale
  return conv


def ddpm_conv3x3(in_planes, out_planes, stride=1, bias=True, dilation=1, init_scale=1., padding=1):
  """
  3x3 convolution with DDPM initialization.
  """
  conv = nn.Conv2d(
    in_planes,
    out_planes,
    kernel_size = 3,
    stride      = stride,
    padding     = padding,
    dilation    = dilation,
    bias        = bias,
  )
  conv.weight.data = default_init(init_scale)(conv.weight.data.shape)
  nn.init.zeros_(conv.bias)
  return conv

  #..........................................................................
  # Functions below are ported from the NCSNv1/NCSNv2 codebase:
  # https://github.com/ermongroup/ncsn
  # https://github.com/ermongroup/ncsnv2
  #..........................................................................

class CRPBlock(nn.Module):
  def __init__(self, features, n_stages, act=nn.ReLU(), maxpool=True):
    super().__init__()
    self.convs = nn.ModuleList()
    for i in range(n_stages):
      self.convs.append(ncsn_conv3x3(features, features, stride=1, bias=False))
    self.n_stages = n_stages
    if maxpool:
      self.pool = nn.MaxPool2d(kernel_size=5, stride=1, padding=2)
    else:
      self.pool = nn.AvgPool2d(kernel_size=5, stride=1, padding=2)

    self.act = act

  def forward(self, x):
    x    = self.act(x)
    path = x
    for i in range(self.n_stages):
      path = self.pool(path)
      path = self.convs[i](path)
      x    = path + x
    return x


class CondCRPBlock(nn.Module):
  def __init__(self, features, n_stages, num_classes, normalizer, act=nn.ReLU()):
    super().__init__()
    self.convs      = nn.ModuleList()
    self.norms      = nn.ModuleList()
    self.normalizer = normalizer
    for i in range(n_stages):
      self.norms.append(normalizer(features, num_classes, bias=True))
      self.convs.append(ncsn_conv3x3(features, features, stride=1, bias=False))

    self.n_stages = n_stages
    self.pool     = nn.AvgPool2d(kernel_size=5, stride=1, padding=2)
    self.act      = act

  def forward(self, x, y):
    x    = self.act(x)
    path = x
    for i in range(self.n_stages):
      path = self.norms[i](path, y)
      path = self.pool(path)
      path = self.convs[i](path)

      x    = path + x
    return x


class RCUBlock(nn.Module):
  def __init__(self, features, n_blocks, n_stages, act=nn.ReLU()):
    super().__init__()

    for i in range(n_blocks):
      for j in range(n_stages):
        setattr(self, '{}_{}_conv'.format(i + 1, j + 1), ncsn_conv3x3(features, features, stride=1, bias=False))

    self.stride   = 1
    self.n_blocks = n_blocks
    self.n_stages = n_stages
    self.act      = act

  def forward(self, x):
    for i in range(self.n_blocks):
      residual = x
      for j in range(self.n_stages):
        x = self.act(x)
        x = getattr(self, '{}_{}_conv'.format(i + 1, j + 1))(x)

      x += residual
    return x


class CondRCUBlock(nn.Module):
  def __init__(self, features, n_blocks, n_stages, num_classes, normalizer, act=nn.ReLU()):
    super().__init__()

    for i in range(n_blocks):
      for j in range(n_stages):
        setattr(self, '{}_{}_norm'.format(i + 1, j + 1), normalizer(features, num_classes, bias=True))
        setattr(self, '{}_{}_conv'.format(i + 1, j + 1), ncsn_conv3x3(features, features, stride=1, bias=False))

    self.stride     = 1
    self.n_blocks   = n_blocks
    self.n_stages   = n_stages
    self.act        = act
    self.normalizer = normalizer

  def forward(self, x, y):
    for i in range(self.n_blocks):
      residual = x
      for j in range(self.n_stages):
        x = getattr(self, '{}_{}_norm'.format(i + 1, j + 1))(x, y)
        x = self.act(x)
        x = getattr(self, '{}_{}_conv'.format(i + 1, j + 1))(x)

      x += residual
    return x


class MSFBlock(nn.Module):
  def __init__(self, in_planes, features):
    super().__init__()
    assert isinstance(in_planes, list) or isinstance(in_planes, tuple)
    self.convs = nn.ModuleList()
    self.features = features

    for i in range(len(in_planes)):
      self.convs.append(ncsn_conv3x3(in_planes[i], features, stride=1, bias=True))

  def forward(self, xs, shape):
    sums = torch.zeros(xs[0].shape[0], self.features, *shape, device=xs[0].device)
    for i in range(len(self.convs)):
      h     = self.convs[i](xs[i])
      h     = F.interpolate(h, size=shape, mode='bilinear', align_corners=True)
      sums += h
    return sums


class CondMSFBlock(nn.Module):
  def __init__(self, in_planes, features, num_classes, normalizer):
    super().__init__()
    assert isinstance(in_planes, list) or isinstance(in_planes, tuple)

    self.convs      = nn.ModuleList()
    self.norms      = nn.ModuleList()
    self.features   = features
    self.normalizer = normalizer

    for i in range(len(in_planes)):
      self.convs.append(ncsn_conv3x3(in_planes[i], features, stride=1, bias=True))
      self.norms.append(normalizer(in_planes[i], num_classes, bias=True))

  def forward(self, xs, y, shape):
    sums = torch.zeros(xs[0].shape[0], self.features, *shape, device=xs[0].device)
    for i in range(len(self.convs)):
      h     = self.norms[i](xs[i], y)
      h     = self.convs[i](h)
      h     = F.interpolate(h, size=shape, mode='bilinear', align_corners=True)
      sums += h
    return sums


class RefineBlock(nn.Module):
  def __init__(self, in_planes, features, act=nn.ReLU(), start=False, end=False, maxpool=True):
    super().__init__()

    assert isinstance(in_planes, tuple) or isinstance(in_planes, list)
    self.n_blocks = n_blocks = len(in_planes)

    self.adapt_convs = nn.ModuleList()
    for i in range(n_blocks):
      self.adapt_convs.append(RCUBlock(in_planes[i], 2, 2, act))

    self.output_convs = RCUBlock(features, 3 if end else 1, 2, act)

    if not start:
      self.msf = MSFBlock(in_planes, features)

    self.crp = CRPBlock(features, 2, act, maxpool=maxpool)

  def forward(self, xs, output_shape):
    assert isinstance(xs, tuple) or isinstance(xs, list)
    hs = []
    for i in range(len(xs)):
      h = self.adapt_convs[i](xs[i])
      hs.append(h)

    if self.n_blocks > 1:
      h = self.msf(hs, output_shape)
    else:
      h = hs[0]

    h = self.crp(h)
    h = self.output_convs(h)

    return h


class CondRefineBlock(nn.Module):
  def __init__(self, in_planes, features, num_classes, normalizer, act=nn.ReLU(), start=False, end=False):
    super().__init__()

    assert isinstance(in_planes, tuple) or isinstance(in_planes, list)
    self.n_blocks = n_blocks = len(in_planes)

    self.adapt_convs = nn.ModuleList()
    for i in range(n_blocks):
      self.adapt_convs.append(
        CondRCUBlock(in_planes[i], 2, 2, num_classes, normalizer, act)
      )

    self.output_convs = CondRCUBlock(features, 3 if end else 1, 2, num_classes, normalizer, act)

    if not start:
      self.msf = CondMSFBlock(in_planes, features, num_classes, normalizer)

    self.crp = CondCRPBlock(features, 2, num_classes, normalizer, act)

  def forward(self, xs, y, output_shape):
    assert isinstance(xs, tuple) or isinstance(xs, list)
    hs = []
    for i in range(len(xs)):
      h = self.adapt_convs[i](xs[i], y)
      hs.append(h)

    if self.n_blocks > 1:
      h = self.msf(hs, y, output_shape)
    else:
      h = hs[0]

    h = self.crp(h, y)
    h = self.output_convs(h, y)

    return h


class ConvMeanPool(nn.Module):
  def __init__(self, input_dim, output_dim, kernel_size=3, biases=True, adjust_padding=False):
    super().__init__()
    if not adjust_padding:
      conv = nn.Conv2d(
        input_dim,
        output_dim,
        kernel_size,
        stride  = 1,
        padding = kernel_size // 2,
        bias    = biases,
      )
      self.conv = conv
    else:
      conv = nn.Conv2d(
        input_dim,
        output_dim,
        kernel_size,
        stride  = 1,
        padding = kernel_size // 2,
        bias    = biases,
      )

      self.conv = nn.Sequential(
        nn.ZeroPad2d((1, 0, 1, 0)),
        conv
      )

  def forward(self, inputs):
    output = self.conv(inputs)
    output = sum([output[:, :, ::2, ::2], output[:, :, 1::2, ::2],
                  output[:, :, ::2, 1::2], output[:, :, 1::2, 1::2]]) / 4.
    return output


class MeanPoolConv(nn.Module):
  def __init__(self, input_dim, output_dim, kernel_size=3, biases=True):
    super().__init__()
    self.conv = nn.Conv2d(
      input_dim, 
      output_dim,
      kernel_size,
      stride  = 1,
      padding = kernel_size // 2,
      bias    = biases,
    )

  def forward(self, inputs):
    output = inputs
    output = sum([output[:, :, ::2, ::2], output[:, :, 1::2, ::2],
                  output[:, :, ::2, 1::2], output[:, :, 1::2, 1::2]]) / 4.
    return self.conv(output)


class UpsampleConv(nn.Module):
  def __init__(self, input_dim, output_dim, kernel_size=3, biases=True):
    super().__init__()
    self.conv = nn.Conv2d(
      input_dim,
      output_dim,
      kernel_size,
      stride  = 1,
      padding = kernel_size // 2,
      bias    = biases,
    )
    self.pixelshuffle = nn.PixelShuffle(upscale_factor=2)

  def forward(self, inputs):
    output = inputs
    output = torch.cat([output, output, output, output], dim=1)
    output = self.pixelshuffle(output)
    return self.conv(output)


class ConditionalResidualBlock(nn.Module):
  def __init__(self, input_dim, output_dim, num_classes, resample=1, act=nn.ELU(),
               normalization=ConditionalInstanceNorm2dPlus, adjust_padding=False, dilation=None):
    super().__init__()
    self.non_linearity = act
    self.input_dim     = input_dim
    self.output_dim    = output_dim
    self.resample      = resample
    self.normalization = normalization
    if resample == 'down':
      if dilation > 1:
        self.conv1      = ncsn_conv3x3(input_dim, input_dim, dilation=dilation)
        self.normalize2 = normalization(input_dim, num_classes)
        self.conv2      = ncsn_conv3x3(input_dim, output_dim, dilation=dilation)
        conv_shortcut   = partial(ncsn_conv3x3, dilation=dilation)
      else:
        self.conv1      = ncsn_conv3x3(input_dim, input_dim)
        self.normalize2 = normalization(input_dim, num_classes)
        self.conv2      = ConvMeanPool(input_dim, output_dim, 3, adjust_padding=adjust_padding)
        conv_shortcut   = partial(ConvMeanPool, kernel_size=1, adjust_padding=adjust_padding)

    elif resample is None:
      if dilation > 1:
        conv_shortcut   = partial(ncsn_conv3x3, dilation=dilation)
        self.conv1      = ncsn_conv3x3(input_dim, output_dim, dilation=dilation)
        self.normalize2 = normalization(output_dim, num_classes)
        self.conv2      = ncsn_conv3x3(output_dim, output_dim, dilation=dilation)
      else:
        conv_shortcut   = nn.Conv2d
        self.conv1      = ncsn_conv3x3(input_dim, output_dim)
        self.normalize2 = normalization(output_dim, num_classes)
        self.conv2      = ncsn_conv3x3(output_dim, output_dim)
    else:
      raise Exception('invalid resample value')

    if output_dim != input_dim or resample is not None:
      self.shortcut = conv_shortcut(input_dim, output_dim)

    self.normalize1 = normalization(input_dim, num_classes)

  def forward(self, x, y):
    output = self.normalize1(x, y)
    output = self.non_linearity(output)
    output = self.conv1(output)
    output = self.normalize2(output, y)
    output = self.non_linearity(output)
    output = self.conv2(output)

    if self.output_dim == self.input_dim and self.resample is None:
      shortcut = x
    else:
      shortcut = self.shortcut(x)

    return shortcut + output


class ResidualBlock(nn.Module):
  def __init__(self, input_dim, output_dim, resample=None, act=nn.ELU(),
               normalization=nn.InstanceNorm2d, adjust_padding=False, dilation=1):
    super().__init__()
    self.non_linearity = act
    self.input_dim     = input_dim
    self.output_dim    = output_dim
    self.resample      = resample
    self.normalization = normalization
    if resample == 'down':
      if dilation > 1:
        self.conv1      = ncsn_conv3x3(input_dim, input_dim, dilation=dilation)
        self.normalize2 = normalization(input_dim)
        self.conv2      = ncsn_conv3x3(input_dim, output_dim, dilation=dilation)
        conv_shortcut   = partial(ncsn_conv3x3, dilation=dilation)
      else:
        self.conv1      = ncsn_conv3x3(input_dim, input_dim)
        self.normalize2 = normalization(input_dim)
        self.conv2      = ConvMeanPool(input_dim, output_dim, 3, adjust_padding=adjust_padding)
        conv_shortcut   = partial(ConvMeanPool, kernel_size=1, adjust_padding=adjust_padding)

    elif resample is None:
      if dilation > 1:
        conv_shortcut   = partial(ncsn_conv3x3, dilation=dilation)
        self.conv1      = ncsn_conv3x3(input_dim, output_dim, dilation=dilation)
        self.normalize2 = normalization(output_dim)
        self.conv2      = ncsn_conv3x3(output_dim, output_dim, dilation=dilation)
      else:
        # conv_shortcut = nn.Conv2d ### Something wierd here.
        conv_shortcut   = partial(ncsn_conv1x1)
        self.conv1      = ncsn_conv3x3(input_dim, output_dim)
        self.normalize2 = normalization(output_dim)
        self.conv2      = ncsn_conv3x3(output_dim, output_dim)
    else:
      raise Exception('invalid resample value')

    if output_dim != input_dim or resample is not None:
      self.shortcut = conv_shortcut(input_dim, output_dim)

    self.normalize1 = normalization(input_dim)

  def forward(self, x):
    output = self.normalize1(x)
    output = self.non_linearity(output)
    output = self.conv1(output)
    output = self.normalize2(output)
    output = self.non_linearity(output)
    output = self.conv2(output)

    if self.output_dim == self.input_dim and self.resample is None:
      shortcut = x
    else:
      shortcut = self.shortcut(x)

    return shortcut + output


#..........................................................................
# Functions below are ported over from the DDPM codebase:
#  https://github.com/hojonathanho/diffusion/blob/master/diffusion_tf/nn.py
#..........................................................................

def get_timestep_embedding(timesteps, embedding_dim, max_positions=10000):
  assert len(timesteps.shape) == 1  # and timesteps.dtype == tf.int32
  half_dim = embedding_dim // 2
  # magic number 10000 is from transformers
  emb = math.log(max_positions) / (half_dim - 1)
  # emb = math.log(2.) / (half_dim - 1)
  emb = torch.exp(torch.arange(half_dim, dtype=torch.float32, device=timesteps.device) * -emb)
  # emb = tf.range(num_embeddings, dtype=jnp.float32)[:, None] * emb[None, :]
  # emb = tf.cast(timesteps, dtype=jnp.float32)[:, None] * emb[None, :]
  emb = timesteps.float()[:, None] * emb[None, :]
  emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=1)
  if embedding_dim % 2 == 1:  # zero pad
    emb = F.pad(emb, (0, 1), mode='constant')
  assert emb.shape == (timesteps.shape[0], embedding_dim)
  return emb

class LinearProjection(nn.Module):
  '''
  Linear projection of a tensor using the dot product between that tensor and 
  the learnable weights 'W' plus a learnable bias 'b'.
  The dot product is taken over the channels dimension.
  '''
  def __init__(self, in_dim, num_units, init_scale=0.1):
    super().__init__()
    # Learnable parameters W and b
    self.W = nn.Parameter(default_init(scale=init_scale)((in_dim, num_units)), requires_grad=True)
    self.b = nn.Parameter(torch.zeros(num_units), requires_grad=True)

  def _einsum_(self, a, b, c, x, y):
    einsum_str = '{},{}->{}'.format(''.join(a), ''.join(b), ''.join(c))
    return torch.einsum(einsum_str, x, y)

  def contract_inner(self, x, y):
    """
    Dot product between x and y, taken over the first dimension of y 
    and the last dimension of x.
    """
    # If 'x' is a 4D tensor,  x_chars = ['a', 'b', 'c', 'd']
    x_chars    = list(string.ascii_lowercase[:len(x.shape)])
    # If 'x' and 'y' are 4D tensors,  y_chars = ['e', 'f', 'g', 'h']
    y_chars    = list(string.ascii_lowercase[len(x.shape):len(y.shape) + len(x.shape)])
    y_chars[0] = x_chars[-1]  
    # out_chars = ['a', 'b', 'c', 'f', 'g', 'h']
    out_chars  = x_chars[:-1] + y_chars[1:]
    # Sum of the products of elements of x and y, where the sum is taken 
    # over first dimension of y and the last dimension of x.
    return self._einsum_(x_chars, y_chars, out_chars, x, y)

  def forward(self, x):
    # Make channels the last dimension of x
    x = x.permute(0, 2, 3, 1)
    # Dot product between x and W, taken over the last dimension of x
    # and the first dimension of W.
    y = self.contract_inner(x, self.W) + self.b
    # Put channels back to the second dimension of y
    return y.permute(0, 3, 1, 2)


class AttentionBlock(nn.Module):
  """
  Channel-wise self-attention block.
  """
  def __init__(self, channels):
    super().__init__()
    self.GroupNorm_0 = nn.GroupNorm(num_groups=32, num_channels=channels, eps=1e-6)
    self.linearproj_0 = LinearProjection(channels, channels)
    self.linearproj_1 = LinearProjection(channels, channels)
    self.linearproj_2 = LinearProjection(channels, channels)
    self.linearproj_3 = LinearProjection(channels, channels, init_scale=0.)

  def forward(self, x):
    B, C, H, W = x.shape
    h = self.GroupNorm_0(x)
    # Obtain q, k, v using linear projections of h 
    q = self.linearproj_0(h) # q = h.W_q, taken over channels dim of h and W_q = [channels, channels]
    k = self.linearproj_1(h) # k = h.W_k, taken over channels dim of h and W_k = [channels, channels]
    v = self.linearproj_2(h) # v = h.W_v, taken over channels dim of h and W_v = [channels, channels]

    # w = q . k^T / sqrt(C), dot product taken over channels dimension of q and k
    w = torch.einsum('bchw,bcij->bhwij', q, k) * (int(C) ** (-0.5))
    w = torch.reshape(w, (B, H, W, H * W))
    # Apply softmax over the last dimension of w
    w = F.softmax(w, dim=-1)
    w = torch.reshape(w, (B, H, W, H, W))
    # h = w . v, dot product (or matrix multiplication) taken over 
    #            the last two dimensions of w and v
    h = torch.einsum('bhwij,bcij->bchw', w, v)
    # Linear projection of h
    h = self.linearproj_3(h)
    # Add input to the output of the attention block
    return x + h


class Upsample(nn.Module):
  def __init__(self, channels, with_conv=False):
    super().__init__()
    if with_conv:
      self.Conv_0 = ddpm_conv3x3(channels, channels)
    self.with_conv = with_conv

  def forward(self, x):
    B, C, H, W = x.shape
    h = F.interpolate(x, (H * 2, W * 2), mode='nearest')
    if self.with_conv:
      h = self.Conv_0(h)
    return h


class Downsample(nn.Module):
  def __init__(self, channels, with_conv=False):
    super().__init__()
    if with_conv:
      self.Conv_0 = ddpm_conv3x3(channels, channels, stride=2, padding=0)
    self.with_conv = with_conv

  def forward(self, x):
    B, C, H, W = x.shape
    # Emulate 'SAME' padding
    if self.with_conv:
      x = F.pad(x, (0, 1, 0, 1))
      x = self.Conv_0(x)
    else:
      x = F.avg_pool2d(x, kernel_size=2, stride=2, padding=0)

    assert x.shape == (B, C, H // 2, W // 2)
    return x


class ResnetBlockDDPM(nn.Module):
  """
  The ResNet Blocks used in DDPM.
  """
  def __init__(self, act, in_ch, out_ch=None, temb_dim=None, conv_shortcut=False, dropout=0.1):
    super().__init__()
    if out_ch is None:
      out_ch = in_ch
    self.GroupNorm_0 = nn.GroupNorm(num_groups=32, num_channels=in_ch, eps=1e-6)
    self.act = act
    self.Conv_0 = ddpm_conv3x3(in_ch, out_ch)
    if temb_dim is not None:
      self.Dense_0 = nn.Linear(temb_dim, out_ch)
      self.Dense_0.weight.data = default_init()(self.Dense_0.weight.data.shape)
      nn.init.zeros_(self.Dense_0.bias)

    self.GroupNorm_1 = nn.GroupNorm(num_groups=32, num_channels=out_ch, eps=1e-6)
    self.Dropout_0   = nn.Dropout(dropout)
    self.Conv_1      = ddpm_conv3x3(out_ch, out_ch, init_scale=0.)
    if in_ch != out_ch:
      if conv_shortcut:
        self.Conv_2 = ddpm_conv3x3(in_ch, out_ch)
      else:
        self.linearproj_0 = LinearProjection(in_ch, out_ch)
    self.out_ch           = out_ch
    self.in_ch            = in_ch
    self.conv_shortcut    = conv_shortcut

  def forward(self, x, temb=None):
    B, C, H, W = x.shape
    assert C == self.in_ch
    out_ch = self.out_ch if self.out_ch else self.in_ch
    h      = self.act(self.GroupNorm_0(x))
    h      = self.Conv_0(h)
    # Add bias to each feature map conditioned on the time embedding
    if temb is not None:
      h += self.Dense_0(self.act(temb))[:, :, None, None]
    h = self.act(self.GroupNorm_1(h))
    h = self.Dropout_0(h)
    h = self.Conv_1(h)
    if C != out_ch:
      if self.conv_shortcut:
        x = self.Conv_2(x)
      else:
        x = self.linearproj_0(x)
    return x + h

## Layers for defining NCSN++

In [ ]:
conv1x1      = ddpm_conv1x1
conv3x3      = ddpm_conv3x3

class GaussianFourierProjection(nn.Module):
  """
  Gaussian Fourier embeddings for noise levels.
  """
  def __init__(self, embedding_size=256, scale=1.0):
    super().__init__()
    self.W = nn.Parameter(torch.randn(embedding_size) * scale, requires_grad=False)

  def forward(self, x):
    x_proj = x[:, None] * self.W[None, :] * 2 * np.pi
    return torch.cat([torch.sin(x_proj), torch.cos(x_proj)], dim=-1)


class Combine(nn.Module):
  """
  Combine information from skip connections.
  """
  def __init__(self, dim1, dim2, method='cat'):
    super().__init__()
    self.Conv_0 = conv1x1(dim1, dim2)
    self.method = method

  def forward(self, x, y):
    h = self.Conv_0(x)
    if self.method == 'cat':
      return torch.cat([h, y], dim=1)
    elif self.method == 'sum':
      return h + y
    else:
      raise ValueError(f'Method {self.method} not recognized.')


class AttentionBlockpp(nn.Module):
  """
  Channel-wise self-attention block. Modified from DDPM.
  """
  def __init__(self, channels, skip_rescale=False, init_scale=0.):
    super().__init__()
    self.GroupNorm_0 = nn.GroupNorm(
      num_groups   = min(channels // 4, 32),
      num_channels = channels,
      eps          = 1e-6,
    )
    self.linearproj_0 = LinearProjection(channels, channels)
    self.linearproj_1 = LinearProjection(channels, channels)
    self.linearproj_2 = LinearProjection(channels, channels)
    self.linearproj_3 = LinearProjection(channels, channels, init_scale=init_scale)
    self.skip_rescale = skip_rescale

  def forward(self, x):
    B, C, H, W = x.shape
    h = self.GroupNorm_0(x)
    q = self.linearproj_0(h)
    k = self.linearproj_1(h)
    v = self.linearproj_2(h)

    w = torch.einsum('bchw,bcij->bhwij', q, k) * (int(C) ** (-0.5))
    w = torch.reshape(w, (B, H, W, H * W))
    w = F.softmax(w, dim=-1)
    w = torch.reshape(w, (B, H, W, H, W))
    h = torch.einsum('bhwij,bcij->bchw', w, v)
    h = self.linearproj_3(h)
    if not self.skip_rescale:
      return x + h
    else:
      return (x + h) / np.sqrt(2.)


class Upsample(nn.Module):
  def __init__(self, in_ch=None, out_ch=None, with_conv=False, fir=False,
               fir_kernel=(1, 3, 3, 1)):
    super().__init__()
    out_ch = out_ch if out_ch else in_ch
    if not fir:
      if with_conv:
        self.Conv_0 = conv3x3(in_ch, out_ch)
    else:
      if with_conv:
        self.Conv2d_0 = Conv2d(
          in_ch,
          out_ch,
          kernel          = 3,
          up              = True,
          resample_kernel = fir_kernel,
          use_bias        = True,
          kernel_init     = default_init(),
        )
    self.fir        = fir
    self.with_conv  = with_conv
    self.fir_kernel = fir_kernel
    self.out_ch     = out_ch

  def forward(self, x):
    B, C, H, W = x.shape
    if not self.fir:
      h = F.interpolate(x, (H * 2, W * 2), 'nearest')
      if self.with_conv:
        h = self.Conv_0(h)
    else:
      if not self.with_conv:
        h = upsample_2d(x, self.fir_kernel, factor=2)
      else:
        h = self.Conv2d_0(x)

    return h


class Downsample(nn.Module):
  def __init__(self, in_ch=None, out_ch=None, with_conv=False, fir=False,
               fir_kernel=(1, 3, 3, 1)):
    super().__init__()
    out_ch = out_ch if out_ch else in_ch
    if not fir:
      if with_conv:
        self.Conv_0 = conv3x3(in_ch, out_ch, stride=2, padding=0)
    else:
      if with_conv:
        self.Conv2d_0 = Conv2d(
          in_ch,
          out_ch,
          kernel          = 3,
          down            = True,
          resample_kernel = fir_kernel,
          use_bias        = True,
          kernel_init     = default_init(),
        )
    self.fir        = fir
    self.fir_kernel = fir_kernel
    self.with_conv  = with_conv
    self.out_ch     = out_ch

  def forward(self, x):
    B, C, H, W = x.shape
    if not self.fir:
      if self.with_conv:
        x = F.pad(x, (0, 1, 0, 1))
        x = self.Conv_0(x)
      else:
        x = F.avg_pool2d(x, 2, stride=2)
    else:
      if not self.with_conv:
        x = downsample_2d(x, self.fir_kernel, factor=2)
      else:
        x = self.Conv2d_0(x)

    return x


class ResnetBlockDDPMpp(nn.Module):
  """
  ResBlock adapted from DDPM.
  """

  def __init__(self, act, in_ch, out_ch=None, temb_dim=None, conv_shortcut=False,
               dropout=0.1, skip_rescale=False, init_scale=0.):
    super().__init__()
    out_ch = out_ch if out_ch else in_ch
    self.GroupNorm_0 = nn.GroupNorm(
      num_groups   = min(in_ch // 4, 32),
      num_channels = in_ch,
      eps          = 1e-6,
    )
    self.Conv_0 = conv3x3(in_ch, out_ch)
    if temb_dim is not None:
      self.Dense_0 = nn.Linear(temb_dim, out_ch)
      self.Dense_0.weight.data = default_init()(self.Dense_0.weight.data.shape)
      nn.init.zeros_(self.Dense_0.bias)
    self.GroupNorm_1 = nn.GroupNorm(
      num_groups   = min(out_ch // 4, 32),
      num_channels = out_ch,
      eps          = 1e-6,
    )
    self.Dropout_0 = nn.Dropout(dropout)
    self.Conv_1    = conv3x3(out_ch, out_ch, init_scale=init_scale)
    if in_ch != out_ch:
      if conv_shortcut:
        self.Conv_2 = conv3x3(in_ch, out_ch)
      else:
        self.linearproj_0 = LinearProjection(in_ch, out_ch)

    self.skip_rescale  = skip_rescale
    self.act           = act
    self.out_ch        = out_ch
    self.conv_shortcut = conv_shortcut

  def forward(self, x, temb=None):
    h = self.act(self.GroupNorm_0(x))
    h = self.Conv_0(h)
    if temb is not None:
      h += self.Dense_0(self.act(temb))[:, :, None, None]
    h = self.act(self.GroupNorm_1(h))
    h = self.Dropout_0(h)
    h = self.Conv_1(h)
    if x.shape[1] != self.out_ch:
      if self.conv_shortcut:
        x = self.Conv_2(x)
      else:
        x = self.linearproj_0(x)
    if not self.skip_rescale:
      return x + h
    else:
      return (x + h) / np.sqrt(2.)


class ResnetBlockBigGANpp(nn.Module):
  def __init__(self, act, in_ch, out_ch=None, temb_dim=None, up=False, down=False,
               dropout=0.1, fir=False, fir_kernel=(1, 3, 3, 1),
               skip_rescale=True, init_scale=0.):
    super().__init__()

    out_ch           = out_ch if out_ch else in_ch
    self.GroupNorm_0 = nn.GroupNorm(num_groups=min(in_ch // 4, 32), num_channels=in_ch, eps=1e-6)
    self.up          = up
    self.down        = down
    self.fir         = fir
    self.fir_kernel  = fir_kernel

    self.Conv_0 = conv3x3(in_ch, out_ch)
    if temb_dim is not None:
      self.Dense_0 = nn.Linear(temb_dim, out_ch)
      self.Dense_0.weight.data = default_init()(self.Dense_0.weight.shape)
      nn.init.zeros_(self.Dense_0.bias)

    self.GroupNorm_1 = nn.GroupNorm(
      num_groups   = min(out_ch // 4, 32),
      num_channels = out_ch,
      eps          = 1e-6,
    )
    self.Dropout_0   = nn.Dropout(dropout)
    self.Conv_1      = conv3x3(out_ch, out_ch, init_scale=init_scale)
    if in_ch != out_ch or up or down:
      self.Conv_2 = conv1x1(in_ch, out_ch)

    self.skip_rescale = skip_rescale
    self.act          = act
    self.in_ch        = in_ch
    self.out_ch       = out_ch

  def forward(self, x, temb=None):
    h = self.act(self.GroupNorm_0(x))

    if self.up:
      if self.fir:
        h = upsample_2d(h, self.fir_kernel, factor=2)
        x = upsample_2d(x, self.fir_kernel, factor=2)
      else:
        h = naive_upsample_2d(h, factor=2)
        x = naive_upsample_2d(x, factor=2)
    elif self.down:
      if self.fir:
        h = downsample_2d(h, self.fir_kernel, factor=2)
        x = downsample_2d(x, self.fir_kernel, factor=2)
      else:
        h = naive_downsample_2d(h, factor=2)
        x = naive_downsample_2d(x, factor=2)

    h = self.Conv_0(h)
    # Add bias to each feature map conditioned on the time embedding
    if temb is not None:
      h += self.Dense_0(self.act(temb))[:, :, None, None]
    h = self.act(self.GroupNorm_1(h))
    h = self.Dropout_0(h)
    h = self.Conv_1(h)

    if self.in_ch != self.out_ch or self.up or self.down:
      x = self.Conv_2(x)

    if not self.skip_rescale:
      return x + h
    else:
      return (x + h) / np.sqrt(2.)

## Models

In [ ]:
# Score model's names to use 
_MODELS             = {}

ResnetBlockDDPM     = ResnetBlockDDPMpp
ResnetBlockBigGAN   = ResnetBlockBigGANpp
default_initializer = default_init

def register_model(cls=None, *, name=None):
    """
    A decorator for registering model classes.
    """

    def _register(cls):
        if name is None:
            local_name = cls.__name__
        else:
            local_name = name
        if local_name in _MODELS:
            raise ValueError(f'Already registered model with name: {local_name}')
        _MODELS[local_name] = cls
        return cls

    if cls is None:
        return _register
    else:
        return _register(cls)

@register_model(name='ncsnpp') 
class NCSNpp(nn.Module):
  """
  NCSN++ model
  """
  def __init__(self, config):
    super().__init__()
    self.config           = config
    self.act              = act              = get_act(config)
    self.register_buffer('sigmas', torch.tensor(self.get_sigmas(config)))

    self.nf               = nf               = config.model.nf
    ch_mult               = config.model.ch_mult
    self.num_res_blocks   = num_res_blocks   = config.model.num_res_blocks
    self.attn_resolutions = attn_resolutions = config.model.attn_resolutions
    dropout               = config.model.dropout
    resamp_with_conv      = config.model.resamp_with_conv
    self.num_resolutions  = num_resolutions  = len(ch_mult)
    self.all_resolutions  = all_resolutions  = [
      config.data.image_size // (2 ** i) for i in range(num_resolutions)
      ]

    self.conditional       = conditional       = config.model.conditional  # noise-conditional
    fir                    = config.model.fir
    fir_kernel             = config.model.fir_kernel
    self.skip_rescale      = skip_rescale      = config.model.skip_rescale
    self.resblock_type     = resblock_type     = config.model.resblock_type.lower()
    self.progressive       = progressive       = config.model.progressive.lower()
    self.progressive_input = progressive_input = config.model.progressive_input.lower()
    self.embedding_type    = embedding_type    = config.model.embedding_type.lower()
    init_scale             = config.model.init_scale
    assert progressive in ['none', 'output_skip', 'residual']
    assert progressive_input in ['none', 'input_skip', 'residual']
    assert embedding_type in ['fourier', 'positional']
    combine_method         = config.model.progressive_combine.lower()
    combiner               = partial(Combine, method=combine_method)

    modules = []
    # timestep/noise_level embedding; only for continuous training
    if embedding_type == 'fourier':
      # Gaussian Fourier features embeddings.
      assert config.training.continuous, "Fourier features are only used for continuous training."

      modules.append(GaussianFourierProjection(
        embedding_size=nf, scale=config.model.fourier_scale
      ))
      embed_dim = 2 * nf

    elif embedding_type == 'positional':
      embed_dim = nf

    else:
      raise ValueError(f'embedding type {embedding_type} unknown.')

    if conditional:
      modules.append(nn.Linear(embed_dim, nf * 4))
      modules[-1].weight.data = default_initializer()(modules[-1].weight.shape)
      nn.init.zeros_(modules[-1].bias)
      modules.append(nn.Linear(nf * 4, nf * 4))
      modules[-1].weight.data = default_initializer()(modules[-1].weight.shape)
      nn.init.zeros_(modules[-1].bias)

    AttnBlock = partial(
      AttentionBlockpp,
      init_scale   = init_scale,
      skip_rescale = skip_rescale,
    )

    Upsample_layer = partial(
      Upsample,
      with_conv  = resamp_with_conv,
      fir        = fir,
      fir_kernel = fir_kernel,
    )

    if progressive == 'output_skip':
      self.pyramid_upsample = Upsample(
        fir        = fir,
        fir_kernel = fir_kernel,
        with_conv  = False,
      )
    elif progressive == 'residual':
      pyramid_upsample = partial(
        Upsample,
        fir        = fir,
        fir_kernel = fir_kernel,
        with_conv  = True,
      )

    Downsample_layer = partial(
      Downsample,
      with_conv  = resamp_with_conv,
      fir        = fir,
      fir_kernel = fir_kernel,
    )

    if progressive_input == 'input_skip':
      self.pyramid_downsample = Downsample(
        fir        = fir,
        fir_kernel = fir_kernel,
        with_conv  = False,
      )
    elif progressive_input == 'residual':
      pyramid_downsample = partial(
        Downsample,
        fir        = fir,
        fir_kernel = fir_kernel,
        with_conv  = True,
      )

    if resblock_type == 'ddpm':
      ResnetBlock = partial(
        ResnetBlockDDPM,
        act          = act,
        dropout      = dropout,
        init_scale   = init_scale,
        skip_rescale = skip_rescale,
        temb_dim     = nf * 4,
      )

    elif resblock_type == 'biggan':
      ResnetBlock = partial(
        ResnetBlockBigGAN,
        act          = act,
        dropout      = dropout,
        fir          = fir,
        fir_kernel   = fir_kernel,
        init_scale   = init_scale,
        skip_rescale = skip_rescale,
        temb_dim     = nf * 4,
      )

    else:
      raise ValueError(f'resblock type {resblock_type} unrecognized.')

    # Downsampling block ..........................................................

    channels = config.data.num_channels
    if progressive_input != 'none':
      input_pyramid_ch = channels

    modules.append(conv3x3(channels, nf))
    hs_c = [nf]

    in_ch = nf
    for i_level in range(num_resolutions):
      # Residual blocks for this resolution
      for i_block in range(num_res_blocks):
        out_ch = nf * ch_mult[i_level]
        modules.append(ResnetBlock(in_ch=in_ch, out_ch=out_ch))
        in_ch = out_ch

        if all_resolutions[i_level] in attn_resolutions:
          modules.append(AttentionBlock(channels=in_ch))
        hs_c.append(in_ch)

      if i_level != num_resolutions - 1:
        if resblock_type == 'ddpm':
          modules.append(Downsample_layer(in_ch=in_ch))
        else:
          modules.append(ResnetBlock(down=True, in_ch=in_ch))

        if progressive_input == 'input_skip':
          modules.append(combiner(dim1=input_pyramid_ch, dim2=in_ch))
          if combine_method == 'cat':
            in_ch *= 2

        elif progressive_input == 'residual':
          modules.append(pyramid_downsample(in_ch=input_pyramid_ch, out_ch=in_ch))
          input_pyramid_ch = in_ch

        hs_c.append(in_ch)

    in_ch = hs_c[-1]
    modules.append(ResnetBlock(in_ch=in_ch))
    modules.append(AttentionBlock(channels=in_ch))
    modules.append(ResnetBlock(in_ch=in_ch))

    pyramid_ch = 0

    # Upsampling block ..........................................................

    for i_level in reversed(range(num_resolutions)):
      for i_block in range(num_res_blocks + 1):
        out_ch = nf * ch_mult[i_level]
        modules.append(ResnetBlock(in_ch=in_ch + hs_c.pop(),
                                   out_ch=out_ch))
        in_ch = out_ch

      if all_resolutions[i_level] in attn_resolutions:
        modules.append(AttentionBlock(channels=in_ch))

      if progressive != 'none':
        if i_level == num_resolutions - 1:
          if progressive == 'output_skip':
            modules.append(nn.GroupNorm(num_groups=min(in_ch // 4, 32),
                                        num_channels=in_ch, eps=1e-6))
            modules.append(conv3x3(in_ch, channels, init_scale=init_scale))
            pyramid_ch = channels
          elif progressive == 'residual':
            modules.append(nn.GroupNorm(num_groups=min(in_ch // 4, 32),
                                        num_channels=in_ch, eps=1e-6))
            modules.append(conv3x3(in_ch, in_ch, bias=True))
            pyramid_ch = in_ch
          else:
            raise ValueError(f'{progressive} is not a valid name.')
        else:
          if progressive == 'output_skip':
            modules.append(nn.GroupNorm(num_groups=min(in_ch // 4, 32),
                                        num_channels=in_ch, eps=1e-6))
            modules.append(conv3x3(in_ch, channels, bias=True, init_scale=init_scale))
            pyramid_ch = channels
          elif progressive == 'residual':
            modules.append(pyramid_upsample(in_ch=pyramid_ch, out_ch=in_ch))
            pyramid_ch = in_ch
          else:
            raise ValueError(f'{progressive} is not a valid name')

      if i_level != 0:
        if resblock_type == 'ddpm':
          modules.append(Upsample_layer(in_ch=in_ch))
        else:
          modules.append(ResnetBlock(in_ch=in_ch, up=True))

    assert not hs_c

    if progressive != 'output_skip':
      modules.append(nn.GroupNorm(num_groups=min(in_ch // 4, 32),
                                  num_channels=in_ch, eps=1e-6))
      modules.append(conv3x3(in_ch, channels, init_scale=init_scale))

    self.all_modules = nn.ModuleList(modules)

  def get_sigmas(self, config):
    """
    Get sigmas, which are the set of noise levels for SMLD.

    Args:
      config: A ConfigDict object parsed from the config file
    Returns:
      sigmas: A numpy array of noise levels.
    """
    sigmas = np.exp(
      np.linspace(
        np.log(config.model.sigma_max),
        np.log(config.model.sigma_min),
        config.model.num_scales
        )
      )

    return sigmas

  def forward(self, x, time_cond):
    # timestep/noise_level embedding; only for continuous training
    modules = self.all_modules
    m_idx   = 0
    if self.embedding_type == 'fourier':
      # Gaussian Fourier features embeddings
      used_sigmas = time_cond
      temb        = modules[m_idx](torch.log(used_sigmas))
      m_idx      += 1

    elif self.embedding_type == 'positional':
      # Sinusoidal positional embeddings
      timesteps   = time_cond
      used_sigmas = self.sigmas[time_cond.long()]
      temb        = get_timestep_embedding(timesteps, self.nf)

    else:
      raise ValueError(f'embedding type {self.embedding_type} unknown.')

    if self.conditional:
      temb   = modules[m_idx](temb)
      m_idx += 1
      temb   = modules[m_idx](self.act(temb))
      m_idx += 1
    else:
      temb = None

    if not self.config.data.centered:
      # If input data is in [0, 1]
      x = 2 * x - 1.

    # Downsampling block ......................................................

    input_pyramid = None
    if self.progressive_input != 'none':
      input_pyramid = x

    hs     = [modules[m_idx](x)]
    m_idx += 1
    for i_level in range(self.num_resolutions):
      # Residual blocks for this resolution
      for i_block in range(self.num_res_blocks):
        h      = modules[m_idx](hs[-1], temb)
        m_idx += 1
        if h.shape[-1] in self.attn_resolutions:
          h = modules[m_idx](h)
          m_idx += 1

        hs.append(h)

      if i_level != self.num_resolutions - 1:
        if self.resblock_type == 'ddpm':
          h      = modules[m_idx](hs[-1])
          m_idx += 1
        else:
          h = modules[m_idx](hs[-1], temb)
          m_idx += 1

        if self.progressive_input == 'input_skip':
          input_pyramid = self.pyramid_downsample(input_pyramid)
          h             = modules[m_idx](input_pyramid, h)
          m_idx        += 1

        elif self.progressive_input == 'residual':
          input_pyramid = modules[m_idx](input_pyramid)
          m_idx        += 1
          if self.skip_rescale:
            input_pyramid = (input_pyramid + h) / np.sqrt(2.)
          else:
            input_pyramid = input_pyramid + h
          h = input_pyramid

        hs.append(h)

    h       = hs[-1]
    h       = modules[m_idx](h, temb)
    m_idx  += 1
    h       = modules[m_idx](h)
    m_idx  += 1
    h       = modules[m_idx](h, temb)
    m_idx  += 1

    pyramid = None

    # Upsampling block ........................................................

    for i_level in reversed(range(self.num_resolutions)):
      for i_block in range(self.num_res_blocks + 1):
        h      = modules[m_idx](torch.cat([h, hs.pop()], dim=1), temb)
        m_idx += 1

      if h.shape[-1] in self.attn_resolutions:
        h      = modules[m_idx](h)
        m_idx += 1

      if self.progressive != 'none':
        if i_level == self.num_resolutions - 1:
          if self.progressive == 'output_skip':
            pyramid = self.act(modules[m_idx](h))
            m_idx  += 1
            pyramid = modules[m_idx](pyramid)
            m_idx  += 1
          elif self.progressive == 'residual':
            pyramid = self.act(modules[m_idx](h))
            m_idx  += 1
            pyramid = modules[m_idx](pyramid)
            m_idx  += 1
          else:
            raise ValueError(f'{self.progressive} is not a valid name.')
        else:
          if self.progressive == 'output_skip':
            pyramid = self.pyramid_upsample(pyramid)
            pyramid_h = self.act(modules[m_idx](h))
            m_idx    += 1
            pyramid_h = modules[m_idx](pyramid_h)
            m_idx    += 1
            pyramid   = pyramid + pyramid_h
          elif self.progressive == 'residual':
            pyramid = modules[m_idx](pyramid)
            m_idx  += 1
            if self.skip_rescale:
              pyramid = (pyramid + h) / np.sqrt(2.)
            else:
              pyramid = pyramid + h
            h = pyramid
          else:
            raise ValueError(f'{self.progressive} is not a valid name')

      if i_level != 0:
        if self.resblock_type == 'ddpm':
          h      = modules[m_idx](h)
          m_idx += 1
        else:
          h      = modules[m_idx](h, temb)
          m_idx += 1

    assert not hs

    if self.progressive == 'output_skip':
      h = pyramid
    else:
      h      = self.act(modules[m_idx](h))
      m_idx += 1
      h      = modules[m_idx](h)
      m_idx += 1

    assert m_idx == len(modules)
    if self.config.model.scale_by_sigma:
      used_sigmas = used_sigmas.reshape((x.shape[0], *([1] * len(x.shape[1:]))))
      h           = h / used_sigmas
      h           = h.to(torch.float32)

    return h

## Exponential Moving Average

In [ ]:
# Partially based on: 
# https://github.com/tensorflow/tensorflow/blob/r1.13/tensorflow/python/training/moving_averages.py

class ExponentialMovingAverage:
  '''
  Maintains exponential moving average of a set of parameters.
  '''
  def __init__(self, model, decay, use_num_updates=True):
    '''
    Args:
      parameters:      Iterable of `torch.nn.Parameter`; usually the result of
                       `model.parameters()`.
      decay:           The exponential decay.
      use_num_updates: Whether to use number of updates when computing averages.
    '''
    if decay < 0.0 or decay > 1.0:
      raise ValueError('Decay must be between 0 and 1')
    self.decay           = decay
    self.num_updates     = 0 if use_num_updates else None
    self.shadow_params   = [
      p.clone().detach() for p in model.parameters() if p.requires_grad
    ]
    self.collected_params = []

  def update(self, parameters):
    '''
    Update currently maintained parameters.

    Call this every time the parameters are updated, such as the result of
    the `optimizer.step()` call.

    Args:
      parameters: Iterable of `torch.nn.Parameter`; usually the same set of
                  parameters used to initialize this object.
    '''
    decay = self.decay
    if self.num_updates is not None:
      self.num_updates += 1
      decay = min(decay, (1 + self.num_updates) / (10 + self.num_updates))
    one_minus_decay = 1.0 - decay
    with torch.no_grad():
      parameters = [p for p in parameters if p.requires_grad]
      for s_param, param in zip(self.shadow_params, parameters):
        s_param.sub_(one_minus_decay * (s_param - param))

  def copy_to(self, model):
    '''
    Copy current parameters into given collection of parameters.

    Args:
      model: Model whose parameters we want to update with the stored moving averages.
    '''
    myparameters = [p for p in model.parameters() if p.requires_grad]
    for s_param, param in zip(self.shadow_params, myparameters):
      if param.requires_grad:
        param.data.copy_(s_param.data)

  def store(self, model):
    '''
    Save the current model parameters for restoring later.

    Args:
      model: Model whose parameters we want to update with the stored moving averages.
    '''
    self.collected_params = [param.clone() for param in model.parameters()]

  def restore(self, model):
    '''
    Restore the parameters stored with the `store` method.
    Useful to validate the model with EMA parameters without affecting the
    original optimization process. Store the parameters before the
    `copy_to` method. After validation (or model saving), use this to
    restore the former parameters.

    Args:
      model: Model whose parameters we want to update with the stored moving averages.
    '''
    for c_param, param in zip(self.collected_params, model.parameters()):
      param.data.copy_(c_param.data)

  def state_dict(self):
    return dict(
      decay            = self.decay,
      num_updates      = self.num_updates,
      shadow_params    = self.shadow_params
      )

  def load_state_dict(self, state_dict):
    self.decay         = state_dict['decay']
    self.num_updates   = state_dict['num_updates']
    self.shadow_params = state_dict['shadow_params']

## Abstract SDE classes, Reverse SDE, and VE/VP SDE

In [ ]:
class SDE(abc.ABC):
  """
  SDE abstract class. Functions are designed for a mini-batch of inputs.
  """

  def __init__(self, N):
    """
    Construct an SDE.

    Args:
      N: number of discretization time steps.
    """
    super().__init__()
    self.N = N

  @property
  @abc.abstractmethod
  def T(self):
    """
    End time of the SDE.
    """
    pass

  @abc.abstractmethod
  def sde(self, x, t):
    pass

  @abc.abstractmethod
  def marginal_prob(self, x, t):
    """
    Parameters to determine the marginal distribution of the SDE, $p_t(x)$.
    """
    pass

  @abc.abstractmethod
  def prior_sampling(self, shape):
    """
    Generate one sample from the prior distribution, $p_T(x)$.
    """
    pass

  @abc.abstractmethod
  def prior_logp(self, z):
    """
    Compute log-density of the prior distribution.

    Useful for computing the log-likelihood via probability flow ODE.

    Args:
      z: latent code
    Returns:
      log probability density
    """
    pass

  def discretize(self, x, t):
    """
    Discretize the SDE in the form: x_{i+1} = x_i + f_i(x_i) + G_i z_i.

    Useful for reverse diffusion sampling and probability flow sampling.
    Defaults to Euler-Maruyama discretization.

    Args:
      x: a torch tensor
      t: a torch float representing the time step (from 0 to `self.T`)

    Returns:
      f, G
    """
    dt               = 1 / self.N
    drift, diffusion = self.sde(x, t)
    f                = drift * dt
    G                = diffusion * torch.sqrt(torch.tensor(dt, device=t.device))

    return f, G

  def reverse(self, score_fn, probability_flow=False):
    """
    Create the reverse-time SDE/ODE.

    Args:
      score_fn: A time-dependent score-based model that takes x and t and returns the score.
      probability_flow: If `True`, create the reverse-time ODE used for probability flow sampling.
    """
    N             = self.N
    T             = self.T
    sde_fn        = self.sde
    discretize_fn = self.discretize

    # Build the class for reverse-time SDE
    class RSDE(self.__class__):
      def __init__(self):
        self.N = N
        self.probability_flow = probability_flow

      @property
      def T(self):
        return T

      def sde(self, x, t):
        """
        Create the drift and diffusion functions for the reverse SDE/ODE.
        """
        drift, diffusion = sde_fn(x, t)

        score            = score_fn(x, t)
        drift            = drift - diffusion[:, None, None, None] ** 2 * \
                           score * (0.5 if self.probability_flow else 1.)
        # Set the diffusion function to zero for ODEs
        diffusion        = 0. if self.probability_flow else diffusion

        return drift, diffusion

      def discretize(self, x, t):
        """
        Create discretized iteration rules for the reverse diffusion sampler.
        """
        f, G  = discretize_fn(x, t)
        rev_f = f - G[:, None, None, None] ** 2 * score_fn(x, t) \
                * (0.5 if self.probability_flow else 1.)
        rev_G = torch.zeros_like(G) if self.probability_flow else G
        return rev_f, rev_G

    return RSDE()


class VPSDE(SDE):
  def __init__(self, beta_min=0.1, beta_max=20, N=1000):
    """
    Construct a Variance Preserving SDE.

    Args:
      beta_min: value of beta(0)
      beta_max: value of beta(1)
      N: number of discretization steps
    """
    super().__init__(N)
    self.beta_0                = beta_min
    self.beta_1                 = beta_max
    self.N                      = N
    self.discrete_betas         = torch.linspace(beta_min / N, beta_max / N, N)
    self.alphas                 = 1. - self.discrete_betas
    self.alphas_cumprod         = torch.cumprod(self.alphas, dim=0)
    self.sqrt_alphas_cumprod    = torch.sqrt(self.alphas_cumprod)
    self.sqrt_1m_alphas_cumprod = torch.sqrt(1. - self.alphas_cumprod)

  @property
  def T(self):
    return 1

  def sde(self, x, t):
    beta_t    = self.beta_0 + t * (self.beta_1 - self.beta_0)
    drift     = -0.5 * beta_t[:, None, None, None] * x
    diffusion = torch.sqrt(beta_t)
    return drift, diffusion

  def marginal_prob(self, x, t):
    log_mean_coeff = -0.25 * t ** 2 * (self.beta_1 - self.beta_0) \
                     - 0.5 * t * self.beta_0
    mean           = torch.exp(log_mean_coeff[:, None, None, None]) * x
    std            = torch.sqrt(1. - torch.exp(2. * log_mean_coeff))
    return mean, std

  def prior_sampling(self, shape):
    return torch.randn(*shape)

  def prior_logp(self, z):
    shape = z.shape
    N     = np.prod(shape[1:])
    logps = -N / 2. * np.log(2 * np.pi) - torch.sum(z ** 2, dim=(1, 2, 3)) / 2.
    return logps

  def discretize(self, x, t):
    """
    DDPM discretization.
    """
    timestep  = (t * (self.N - 1) / self.T).long()
    beta      = self.discrete_betas.to(x.device)[timestep]
    alpha     = self.alphas.to(x.device)[timestep]
    sqrt_beta = torch.sqrt(beta)
    f         = torch.sqrt(alpha)[:, None, None, None] * x - x
    G         = sqrt_beta
    return f, G


class subVPSDE(SDE):
  def __init__(self, beta_min=0.1, beta_max=20, N=1000):
    """
    Construct the sub-VP SDE that excels at likelihoods.

    Args:
      beta_min: value of beta(0)
      beta_max: value of beta(1)
      N: number of discretization steps
    """
    super().__init__(N)
    self.beta_0 = beta_min
    self.beta_1 = beta_max
    self.N      = N

  @property
  def T(self):
    return 1

  def sde(self, x, t):
    beta_t    = self.beta_0 + t * (self.beta_1 - self.beta_0)
    drift     = -0.5 * beta_t[:, None, None, None] * x
    discount  = 1. - torch.exp(-2 * self.beta_0 * t - \
                (self.beta_1 - self.beta_0) * t ** 2)
    diffusion = torch.sqrt(beta_t * discount)
    return drift, diffusion

  def marginal_prob(self, x, t):
    log_mean_coeff = -0.25 * t ** 2 * (self.beta_1 - self.beta_0) \
                     - 0.5 * t * self.beta_0
    mean           = torch.exp(log_mean_coeff)[:, None, None, None] * x
    std            = 1 - torch.exp(2. * log_mean_coeff)
    return mean, std

  def prior_sampling(self, shape):
    return torch.randn(*shape)

  def prior_logp(self, z):
    shape = z.shape
    N     = np.prod(shape[1:])
    return -N / 2. * np.log(2 * np.pi) - torch.sum(z ** 2, dim=(1, 2, 3)) / 2.


class VESDE(SDE):
  def __init__(self, sigma_min=0.01, sigma_max=50, N=1000):
    """
    Construct a Variance Exploding SDE.

    Args:
      sigma_min: smallest sigma.
      sigma_max: largest sigma.
      N:         number of discretization steps.
    """
    super().__init__(N)
    self.sigma_min       = sigma_min
    self.sigma_max       = sigma_max
    self.discrete_sigmas = torch.exp(
      torch.linspace(
        np.log(self.sigma_min),
        np.log(self.sigma_max), 
        N
      )
    )
    self.N = N

  @property
  def T(self):
    return 1

  def sde(self, x, t):
    sigma     = self.sigma_min * (self.sigma_max / self.sigma_min) ** t
    drift     = torch.zeros_like(x)
    diffusion = sigma * torch.sqrt(
      torch.tensor(
        2 * (np.log(self.sigma_max) - np.log(self.sigma_min)),
        device=t.device,
      )
    )
    return drift, diffusion

  def marginal_prob(self, x, t):
    std  = self.sigma_min * (self.sigma_max / self.sigma_min) ** t
    mean = x
    return mean, std

  def prior_sampling(self, shape):
    return torch.randn(*shape) * self.sigma_max

  def prior_logp(self, z):
    shape = z.shape
    N     = np.prod(shape[1:])
    return -N / 2. * np.log(2 * np.pi * self.sigma_max ** 2) - \
           torch.sum(z ** 2, dim=(1, 2, 3)) / (2 * self.sigma_max ** 2)

  def discretize(self, x, t):
    """
    SMLD (NCSN) discretization.
    """
    self.discrete_sigmas = self.discrete_sigmas.to(t.device)
    timestep       = (t * (self.N - 1) / self.T).long()
    sigma          = self.discrete_sigmas[timestep]
    adjacent_sigma = torch.where(
      timestep == 0,
      torch.zeros_like(t),
      self.discrete_sigmas[timestep - 1]
    )
    f              = torch.zeros_like(x)
    G              = torch.sqrt(sigma ** 2 - adjacent_sigma ** 2)
    return f, G

## Functions for Model Definition

In [ ]:
class ScoreModel(torch.nn.Module):

  def __init__(self, config):
    super().__init__()

    self.config                  = config

    # Create a score model given the specified configuration.
    
    model_name                   = config.model.name # for example 'ncsnpp'
    self.model                   = self.get_model(model_name)(config)
    self.model                   = self.model.to(config.device)
    #self.model                   = torch.nn.DataParallel(self.model)

    self.num_diffusion_timesteps = 1000
    self.betas                   = None
    self.alphas                  = None
    self.alphas_cumprod          = None
    self.sqrt_alphas_cumprod     = None
    self.sqrt_1m_alphas_cumprod  = None
    self.beta_min                = None
    self.beta_max                = None

  def get_model(self, name):
    return _MODELS[name]

  def get_ddpm_params(self, config):
    """
    Get betas and alphas, which are the parameters used in the original DDPM paper.
    beta  = noise variance applied in a diffusion step.
    alpha = 1 - beta.
    """
    self.num_diffusion_timesteps = 1000
    # These parameters need to be adapted if number of time steps differs from 1000
    beta_start = config.model.beta_min / config.model.num_scales
    beta_end   = config.model.beta_max / config.model.num_scales
    self.betas = np.linspace(
      beta_start,
      beta_end,
      self.num_diffusion_timesteps,
      dtype = np.float32
      )

    self.alphas                 = 1. - self.betas
    self.alphas_cumprod         = np.cumprod(self.alphas, axis=0)
    self.sqrt_alphas_cumprod    = np.sqrt(self.alphas_cumprod)
    self.sqrt_1m_alphas_cumprod = np.sqrt(1. - self.alphas_cumprod)
    self.beta_min               = beta_start * (self.num_diffusion_timesteps - 1)
    self.beta_max               = beta_end * (self.num_diffusion_timesteps - 1)
    return {
      'betas':                   self.betas,
      'alphas':                  self.alphas,
      'alphas_cumprod':          self.alphas_cumprod,
      'sqrt_alphas_cumprod':     self.sqrt_alphas_cumprod,
      'sqrt_1m_alphas_cumprod':  self.sqrt_1m_alphas_cumprod,
      'beta_min':                self.beta_min,
      'beta_max':                self.beta_max,
      'num_diffusion_timesteps': self.num_diffusion_timesteps
    }

  def get_model_fn(self, mymodel, train=False):
    """
    Creates a function to give the output of the score-based model.

    Args:
      mymodel: Model used for computing the scores. 
      train:   `True` for training and `False` for evaluation.

    Returns:
      A model function.
    """

    def model_fn_internal(x, labels):
      """
      Computes the output of the score-based model.

      Args:
        x:      A mini-batch of input data.
        labels: A mini-batch of conditioning variables for time steps.
                Should be interpreted differently for different models.

      Returns:
        A tuple (model output, new mutable states).
      """
      if not train:
        mymodel.eval()
        return mymodel(x, labels)
      else:
        mymodel.train()
        return mymodel(x, labels)

    return model_fn_internal

  def get_score_fn(self, sde, mymodel, train=False, continuous=False):
    """
    Wraps `score_fn` so that the model output corresponds to a real time-dependent score function.

    Args:
      sde:         An `SDE` object that represents the forward SDE.
      mymodel:     Model used for computing the scores. 
      train:      `True` for training and `False` for evaluation.
      continuous:  If `True`, the score-based model is expected to directly take continuous time steps.

    Returns:
      A score function.
    """
    model_fn = self.get_model_fn(mymodel, train=train)

    if isinstance(sde, VPSDE) or isinstance(sde, subVPSDE):
      def score_fn_internal(x, t):
        # Scale neural network output by standard deviation and flip sign
        if continuous or isinstance(sde, subVPSDE):
          # For VP-trained models, t=0 corresponds to the lowest noise level
          # The maximum value of time embedding is assumed to 999 for
          # continuously-trained models.
          labels = t * 999
          score  = model_fn(x, labels)
          std    = sde.marginal_prob(torch.zeros_like(x), t)[1]
        else:
          # For VP-trained models, t=0 corresponds to the lowest noise level
          labels = t * (sde.N - 1)
          score  = model_fn(x, labels)
          std    = sde.sqrt_1m_alphas_cumprod.to(labels.device)[labels.long()]

        score = -score / std[:, None, None, None]
        return score

    elif isinstance(sde, VESDE):
      def score_fn_internal(x, t):
        if continuous:
          labels = sde.marginal_prob(torch.zeros_like(x), t)[1]
        else:
          # For VE-trained models, t=0 corresponds to the highest noise level
          labels  = sde.T - t
          labels *= sde.N - 1
          labels  = torch.round(labels).long()

        score = model_fn(x, labels)

        return score

    else:
      raise NotImplementedError(f"SDE class {sde.__class__.__name__} not yet supported.")

    return score_fn_internal


## Functions related to loss computation and optimization

In [ ]:
class ScoreModelOptimizer(torch.optim.Optimizer):
  '''
  Optimizer class for score model.
  '''
  def __init__(self, config, model):
    self.optimize_fn = None

    if config.optim.optimizer == 'Adam':
      self.optimizer = optim.Adam(
        model.parameters(),
        lr           = config.optim.lr,
        betas        = (config.optim.beta1, 0.999),
        eps          = config.optim.eps,
        weight_decay = config.optim.weight_decay,
      )
    else:
      raise NotImplementedError(
        f'Optimizer {config.optim.optimizer} not supported yet!')

  def optimization_manager(self, config):
    """
    Returns an optimize_fn based on `config`.
    """

    def optimize_fn_internal(
      myoptimizer,
      model,
      step,
      lr        = config.optim.lr,
      warmup    = config.optim.warmup,
      grad_clip = config.optim.grad_clip,
      ):
      """
      Optimizes with warmup and gradient clipping (disabled if negative).
      """
      if warmup > 0:
        for g in myoptimizer.param_groups:
          g['lr'] = lr * np.minimum(step / warmup, 1.0)
      if grad_clip >= 0:
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=grad_clip)
      myoptimizer.step()

    self.optimize_fn = optimize_fn_internal
    return self.optimize_fn


## Sampling Methods

In [ ]:
_CORRECTORS = {}
_PREDICTORS = {}

def register_predictor(cls=None, *, name=None):
  """
  A decorator for registering predictor classes.
  """

  def _register(cls):
    if name is None:
      local_name = cls.__name__
    else:
      local_name = name
    if local_name in _PREDICTORS:
      raise ValueError(f'Already registered model with name: {local_name}')
    _PREDICTORS[local_name] = cls
    return cls

  if cls is None:
    return _register
  else:
    return _register(cls)


def register_corrector(cls=None, *, name=None):
  """
  A decorator for registering corrector classes.
  """

  def _register(cls):
    if name is None:
      local_name = cls.__name__
    else:
      local_name = name
    if local_name in _CORRECTORS:
      raise ValueError(f'Already registered model with name: {local_name}')
    _CORRECTORS[local_name] = cls
    return cls

  if cls is None:
    return _register
  else:
    return _register(cls)


def get_predictor(name):
  return _PREDICTORS[name]


def get_corrector(name):
  return _CORRECTORS[name]


def get_sampling_fn(SModel, model, config, sde, shape, inverse_scaler, eps):
  """
  Create a sampling function.

  Args:
    SModel: An instance of the score model class.
    model:  The true model in SModel.
    config: A `ml_collections.ConfigDict` object that contains all configuration information.
    sde:    A `sde_lib.SDE` object that represents the forward SDE.
    shape:  A sequence of integers representing the expected shape of a single sample.
    inverse_scaler: The inverse data normalizer function.
    eps:    A `float` number. The reverse-time SDE is only integrated to `eps` for numerical stability.

  Returns:
    A function that takes random states and a replicated training state and outputs samples with the
      trailing dimensions matching `shape`.
  """

  sampler_name = config.sampling.method

  # Probability flow ODE sampling with black-box ODE solvers
  if sampler_name.lower() == 'ode':
    sampling_fn = get_ode_sampler(
      SModel         = SModel,
      model          = model,
      sde            = sde,
      shape          = shape,
      inverse_scaler = inverse_scaler,
      denoise        = config.sampling.noise_removal,
      eps            = eps,
      device         = config.device,
    )
  # Predictor-Corrector sampling. Predictor-only and Corrector-only samplers are special cases.
  elif sampler_name.lower() == 'pc':
    predictor   = get_predictor(config.sampling.predictor.lower())
    corrector   = get_corrector(config.sampling.corrector.lower())
    sampling_fn = get_pc_sampler(
      SModel           = SModel,
      model            = model,
      sde              = sde,
      shape            = shape,
      predictor        = predictor,
      corrector        = corrector,
      inverse_scaler   = inverse_scaler,
      snr              = config.sampling.snr,
      n_steps          = config.sampling.n_steps_each,
      probability_flow = config.sampling.probability_flow,
      continuous       = config.training.continuous,
      denoise          = config.sampling.noise_removal,
      eps              = eps,
      device           = config.device,
    )
  else:
    raise ValueError(f"Sampler name {sampler_name} unknown.")

  return sampling_fn


class Predictor(abc.ABC):
  """
  The abstract class for a predictor algorithm.
  """

  def __init__(self, sde, score_fn, probability_flow=False):
    super().__init__()
    self.sde      = sde
    # Compute the reverse SDE/ODE
    self.rsde     = sde.reverse(score_fn, probability_flow)
    self.score_fn = score_fn

  @abc.abstractmethod
  def update_fn(self, x, t):
    """
    One update of the predictor.

    Args:
      x: A PyTorch tensor representing the current state
      t: A Pytorch tensor representing the current time step.

    Returns:
      x: A PyTorch tensor of the next state.
      x_mean: A PyTorch tensor. The next state without random noise. Useful for denoising.
    """
    pass


class Corrector(abc.ABC):
  """
  The abstract class for a corrector algorithm.
  """

  def __init__(self, sde, score_fn, snr, n_steps):
    super().__init__()
    self.sde      = sde
    self.score_fn = score_fn
    self.snr      = snr
    self.n_steps  = n_steps

  @abc.abstractmethod
  def update_fn(self, x, t):
    """
    One update of the corrector.

    Args:
      x: A PyTorch tensor representing the current state
      t: A PyTorch tensor representing the current time step.

    Returns:
      x: A PyTorch tensor of the next state.
      x_mean: A PyTorch tensor. The next state without random noise. Useful for denoising.
    """
    pass


@register_predictor(name='euler_maruyama')
class EulerMaruyamaPredictor(Predictor):
  def __init__(self, sde, score_fn, probability_flow=False):
    super().__init__(sde, score_fn, probability_flow)

  def update_fn(self, x, t):
    dt     = -1. / self.rsde.N
    z      = torch.randn_like(x)
    drift, diffusion = self.rsde.sde(x, t)
    x_mean = x + drift * dt
    x      = x_mean + diffusion[:, None, None, None] * np.sqrt(-dt) * z
    return x, x_mean


@register_predictor(name='reverse_diffusion')
class ReverseDiffusionPredictor(Predictor):
  def __init__(self, sde, score_fn, probability_flow=False):
    super().__init__(sde, score_fn, probability_flow)

  def update_fn(self, x, t):
    f, G   = self.rsde.discretize(x, t)
    z      = torch.randn_like(x)
    x_mean = x - f
    x      = x_mean + G[:, None, None, None] * z
    return x, x_mean


@register_predictor(name='ancestral_sampling')
class AncestralSamplingPredictor(Predictor):
  """
  The ancestral sampling predictor. Currently only supports VE/VP SDEs.
  """

  def __init__(self, sde, score_fn, probability_flow=False):
    super().__init__(sde, score_fn, probability_flow)
    if not isinstance(sde, VPSDE) and not isinstance(sde, VESDE):
      raise NotImplementedError(f"SDE class {sde.__class__.__name__} not yet supported.")
    assert not probability_flow, "Probability flow not supported by ancestral sampling"

  def vesde_update_fn(self, x, t):
    sde    = self.sde
    timestep = (t * (sde.N - 1) / sde.T).long()
    sigma  = sde.discrete_sigmas[timestep]
    adjacent_sigma = torch.where(timestep == 0, torch.zeros_like(t), sde.discrete_sigmas.to(t.device)[timestep - 1])
    score  = self.score_fn(x, t)
    x_mean = x + score * (sigma ** 2 - adjacent_sigma ** 2)[:, None, None, None]
    std    = torch.sqrt((adjacent_sigma ** 2 * (sigma ** 2 - adjacent_sigma ** 2)) / (sigma ** 2))
    noise  = torch.randn_like(x)
    x      = x_mean + std[:, None, None, None] * noise
    return x, x_mean

  def vpsde_update_fn(self, x, t):
    sde      = self.sde
    timestep = (t * (sde.N - 1) / sde.T).long()
    beta     = sde.discrete_betas.to(t.device)[timestep]
    score    = self.score_fn(x, t)
    x_mean   = (x + beta[:, None, None, None] * score) / torch.sqrt(1. - beta)[:, None, None, None]
    noise    = torch.randn_like(x)
    x        = x_mean + torch.sqrt(beta)[:, None, None, None] * noise
    return x, x_mean

  def update_fn(self, x, t):
    if isinstance(self.sde, VESDE):
      return self.vesde_update_fn(x, t)
    elif isinstance(self.sde, VPSDE):
      return self.vpsde_update_fn(x, t)


@register_predictor(name='none')
class NonePredictor(Predictor):
  """
  An empty predictor that does nothing.
  """

  def __init__(self, sde, score_fn, probability_flow=False):
    pass

  def update_fn(self, x, t):
    return x, x


@register_corrector(name='langevin')
class LangevinCorrector(Corrector):
  def __init__(self, sde, score_fn, snr, n_steps):
    super().__init__(sde, score_fn, snr, n_steps)
    if not isinstance(sde, VPSDE) \
        and not isinstance(sde, VESDE) \
        and not isinstance(sde, subVPSDE):
      raise NotImplementedError(f"SDE class {sde.__class__.__name__} not yet supported.")

  def update_fn(self, x, t):
    sde        = self.sde
    score_fn   = self.score_fn
    n_steps    = self.n_steps
    target_snr = self.snr
    if isinstance(sde, VPSDE): # or isinstance(sde, subVPSDE): # AJE: ALTERED THIS LINE BECAUSE subVPSDE DOES NOT HAVE 'alphas'
      timestep = (t * (sde.N - 1) / sde.T).long()
      alpha    = sde.alphas.to(t.device)[timestep]
    else:
      alpha = torch.ones_like(t)

    for i in range(n_steps):
      grad       = score_fn(x, t)
      noise      = torch.randn_like(x)
      grad_norm  = torch.norm(grad.reshape(grad.shape[0], -1), dim=-1).mean()
      noise_norm = torch.norm(noise.reshape(noise.shape[0], -1), dim=-1).mean()
      step_size  = (target_snr * noise_norm / grad_norm) ** 2 * 2 * alpha
      x_mean     = x + step_size[:, None, None, None] * grad
      x          = x_mean + torch.sqrt(step_size * 2)[:, None, None, None] * noise

    return x, x_mean


@register_corrector(name='ald')
class AnnealedLangevinDynamics(Corrector):
  """
  The original annealed Langevin dynamics predictor in NCSN/NCSNv2.

  We include this corrector only for completeness. It was not directly used in our paper.
  """

  def __init__(self, sde, score_fn, snr, n_steps):
    super().__init__(sde, score_fn, snr, n_steps)
    if not isinstance(sde, VPSDE) \
        and not isinstance(sde, VESDE) \
        and not isinstance(sde, subVPSDE):
      raise NotImplementedError(f"SDE class {sde.__class__.__name__} not yet supported.")

  def update_fn(self, x, t):
    sde        = self.sde
    score_fn   = self.score_fn
    n_steps    = self.n_steps
    target_snr = self.snr
    if isinstance(sde, VPSDE): # or isinstance(sde, subVPSDE) <- ALTERED BECAUSE subVPSDE DOES NOT HAVE 'alphas'
      timestep = (t * (sde.N - 1) / sde.T).long()
      alpha    = sde.alphas.to(t.device)[timestep]
    else:
      alpha    = torch.ones_like(t)

    std        = self.sde.marginal_prob(x, t)[1]

    for i in range(n_steps):
      grad      = score_fn(x, t)
      noise     = torch.randn_like(x)
      step_size = (target_snr * std) ** 2 * 2 * alpha
      x_mean    = x + step_size[:, None, None, None] * grad
      x         = x_mean + noise * torch.sqrt(step_size * 2)[:, None, None, None]

    return x, x_mean


@register_corrector(name='none')
class NoneCorrector(Corrector):
  """
  An empty corrector that does nothing.
  """

  def __init__(self, sde, score_fn, snr, n_steps):
    pass

  def update_fn(self, x, t):
    return x, x


def shared_predictor_update_fn(SModel, model, x, t, sde, predictor, probability_flow, continuous):
  """
  A wrapper that configures and returns the update function of predictors.
  """
  score_fn = SModel.get_score_fn(sde, model, train=False, continuous=continuous)
  if predictor is None:
    # Corrector-only sampler
    predictor_obj = NonePredictor(sde, score_fn, probability_flow)
  else:
    predictor_obj = predictor(sde, score_fn, probability_flow)
  return predictor_obj.update_fn(x, t)


def shared_corrector_update_fn(SModel, model, x, t, sde, corrector, continuous, snr, n_steps):
  """
  A wrapper that configures and returns the update function of correctors.
  """
  score_fn = SModel.get_score_fn(sde, model, train=False, continuous=continuous)
  if corrector is None:
    # Predictor-only sampler
    corrector_obj = NoneCorrector(sde, score_fn, snr, n_steps)
    # ###############################################################################
    # print(f'shared_corrector_update_fn: executed "NoneCorrector"')
    # ###############################################################################
  else:
    corrector_obj = corrector(sde, score_fn, snr, n_steps)
    # ###############################################################################
    # print(f'shared_corrector_update_fn: executed "corrector"')
    # ###############################################################################
  return corrector_obj.update_fn(x, t)


def get_pc_sampler(
  SModel,
  model,
  sde,
  shape,
  predictor,
  corrector,
  inverse_scaler,
  snr,
  n_steps          = 1,
  probability_flow = False,
  continuous       = False,
  denoise          = True,
  eps              = 1e-3,
  device           = 'cuda',
  ):
  """
  Create a Predictor-Corrector (PC) sampler.

  Args:
    sde:              An `sde_lib.SDE` object representing the forward SDE.
    shape:            A sequence of integers. The expected shape of a single sample.
    predictor:        A subclass of `sampling.Predictor` representing the predictor algorithm.
    corrector:        A subclass of `sampling.Corrector` representing the corrector algorithm.
    inverse_scaler:   The inverse data normalizer.
    snr:              A `float` number. The signal-to-noise ratio for configuring correctors.
    n_steps:          An integer. The number of corrector steps per predictor update.
    probability_flow: If `True`, solve the reverse-time probability flow ODE when running the predictor.
    continuous:       `True` indicates that the score model was continuously trained.
    denoise:          If `True`, add one-step denoising to the final samples.
    eps:              A `float` number. The reverse-time SDE and ODE are integrated 
                      to `epsilon` to avoid numerical issues.
    device:           PyTorch device.

  Returns:
    A sampling function that returns samples and the number of function evaluations during sampling.
  """
  # Create predictor & corrector update functions
  predictor_update_fn = partial(
    shared_predictor_update_fn,
    sde              = sde,
    predictor        = predictor,
    probability_flow = probability_flow,
    continuous       = continuous,
    )
  
  corrector_update_fn = partial(
    shared_corrector_update_fn,
    sde        = sde,
    corrector  = corrector,
    continuous = continuous,
    snr        = snr,
    n_steps    = n_steps,
    )

  def pc_sampler(SModel, model):
    """
    The PC sampler function.

    Returns:
      Samples, number of function evaluations.
    """
    with torch.no_grad():
      # Initial sample
      x         = sde.prior_sampling(shape).to(device)
      timesteps = torch.linspace(sde.T, eps, sde.N, device=device)

      for i in range(sde.N):
        t         = timesteps[i]
        vec_t     = torch.ones(shape[0], device=t.device) * t

        # ###############################################################################
        # print(f'pc_sampler: SDE type is {type(sde)}')
        # ###############################################################################

        x, x_mean = corrector_update_fn(SModel, model, x, vec_t)
        #print(f'/', end='')

        x, x_mean = predictor_update_fn(SModel, model, x, vec_t)
        #print(f'\\', end='')

        print(f'.', end='')

      return inverse_scaler(x_mean if denoise else x), sde.N * (n_steps + 1)

  return pc_sampler


def get_ode_sampler(
  SModel,
  model,
  sde, 
  shape,
  inverse_scaler,
  denoise = False,
  rtol    = 1e-5,
  atol    = 1e-5,
  method  = 'RK45',
  eps     = 1e-3,
  device  = 'cuda',
  ):
  """
  Probability flow ODE sampler with the black-box ODE solver.

  Args:
    SModel:          An instance of the score model class.
    model:           The true model in SModel.
    sde:             An `sde_lib.SDE` object that represents the forward SDE.
    shape:           A sequence of integers. The expected shape of a single sample.
    inverse_scaler:  The inverse data normalizer.
    denoise:         If `True`, add one-step denoising to final samples.
    rtol:            A `float` number. The relative tolerance level of the ODE solver.
    atol:            A `float` number. The absolute tolerance level of the ODE solver.
    method:          A `str`. The algorithm used for the black-box ODE solver.
                     See the documentation of `scipy.integrate.solve_ivp`.
    eps:             A `float` number. The reverse-time SDE/ODE will be integrated 
                     to `eps` for numerical stability.
    device:          PyTorch device.

  Returns:
    A sampling function that returns samples and
    th number of function evaluations during sampling.
  """

  def denoise_update_fn(SModel, model, x):
    score_fn      = SModel.get_score_fn(sde, model, train=False, continuous=True)
    # Reverse diffusion predictor for denoising
    predictor_obj = ReverseDiffusionPredictor(sde, score_fn, probability_flow=False)
    vec_eps       = torch.ones(x.shape[0], device=x.device) * eps
    _, x          = predictor_obj.update_fn(x, vec_eps)
    return x

  def drift_fn(SModel, model, x, t):
    """
    Get the drift function of the reverse-time SDE.
    """
    score_fn = SModel.get_score_fn(sde, model, train=False, continuous=True)
    rsde     = sde.reverse(score_fn, probability_flow=True)
    return rsde.sde(x, t)[0]

  def ode_sampler(SModel, model, z=None):
    """
    The probability flow ODE sampler with black-box ODE solver.

    Args:
      SModel: An instance of the score model class.
      model:  The true model in SModel.
      z:      If present, generate samples from latent code `z`.
    Returns:
      Tuple with (samples, number of function evaluations).
    """
    with torch.no_grad():
      # Initial sample
      if z is None:
        # If not represent, sample the latent code from the prior distribution of the SDE.
        x = sde.prior_sampling(shape).to(device)
      else:
        x = z

      def ode_func(t, x):
        x     = from_flattened_numpy(x, shape).to(device).type(torch.float32)
        vec_t = torch.ones(shape[0], device=x.device) * t
        drift = drift_fn(SModel, model, x, vec_t)
        return to_flattened_numpy(drift)

      # Black-box ODE solver for the probability flow ODE
      solution = integrate.solve_ivp(
        ode_func,
        (sde.T, eps),
        to_flattened_numpy(x),
        rtol   = rtol,
        atol   = atol,
        method = method,
      )
      nfe = solution.nfev
      x   = torch.tensor(solution.y[:, -1]).reshape(shape).to(device).type(torch.float32)

      # Denoising is equivalent to running one predictor step without adding noise
      if denoise:
        x = denoise_update_fn(SModel, model, x)

      x = inverse_scaler(x)
      return x, nfe

  return ode_sampler

## Likelihood Calculation

In [ ]:
def get_div_fn(fn):
  """
  Create the divergence function of `fn` using the Hutchinson-Skilling trace estimator.
  """

  def div_fn(x, t, eps):
    with torch.enable_grad():
      x.requires_grad_(True)
      fn_eps      = torch.sum(fn(x, t) * eps)
      grad_fn_eps = torch.autograd.grad(fn_eps, x)[0]
    x.requires_grad_(False)
    return torch.sum(grad_fn_eps * eps, dim=tuple(range(1, len(x.shape))))

  return div_fn


def get_likelihood_fn(
    SModel,
    model,
    sde,
    inverse_scaler,
    hutchinson_type = 'Rademacher',
    rtol            = 1e-5,
    atol            = 1e-5,
    method          = 'RK45',
    eps             = 1e-5,
  ):
  """
  Create a function to compute the unbiased log-likelihood estimate of a given data point.

  Args:
    SModel:          An instance of the score model class.
    model:           The true model in SModel.
    sde:             A `sde_lib.SDE` object that represents the forward SDE.
    inverse_scaler:  The inverse data normalizer.
    hutchinson_type: "Rademacher" or "Gaussian".
                     The type of noise for Hutchinson-Skilling trace estimator.
    rtol:            A `float` number. The relative tolerance level of the black-box ODE solver.
    atol:            A `float` number. The absolute tolerance level of the black-box ODE solver.
    method:          The algorithm for the black-box ODE solver.
                     (See documentation about `scipy.integrate.solve_ivp`)
    eps:             A `float` number. The probability flow ODE is integrated to `eps` 
                     for numerical stability.
  Returns:
      A function that given a batch of data points returns ...
         - the log-likelihoods in bits/dim,
         - the latent code, 
         - the number of function evaluations cost by computation.
  """

  def drift_fn(SModel, model, x, t):
    """
    The drift function of the reverse-time SDE.
    """
    score_fn = SModel.get_score_fn(sde, model, train=False, continuous=True)
    # Probability flow ODE is a special case of Reverse SDE
    rsde = sde.reverse(score_fn, probability_flow=True)
    return rsde.sde(x, t)[0]

  def div_fn(SModel, model, x, t, noise):
    return get_div_fn(lambda xx, tt: drift_fn(SModel, model, xx, tt))(x, t, noise)

  def likelihood_fn(SModel, model, data):
    """
    Compute an unbiased estimate to the log-likelihood in bits/dim.

    Args:
      SModel: An instance of the score model class.
      model:  The true model in SModel.
      data:   A PyTorch tensor.

    Returns:
      bpd: A PyTorch tensor of shape [batch size]. The log-likelihoods on `data` in bits/dim.
      z:   A PyTorch tensor of the same shape as `data`. The latent representation of `data` 
           under the probability flow ODE.
      nfe: An integer. The number of function evaluations used for running the black-box ODE solver.
    """
    with torch.no_grad():
      shape = data.shape
      if hutchinson_type == 'Gaussian':
        epsilon = torch.randn_like(data)
      elif hutchinson_type == 'Rademacher':
        epsilon = torch.randint_like(data, low=0, high=2).float() * 2 - 1.
      else:
        raise NotImplementedError(f"Hutchinson type {hutchinson_type} unknown.")

      def ode_func(t, x):
        sample    = from_flattened_numpy(x[:-shape[0]], shape).to(data.device).type(torch.float32)
        vec_t     = torch.ones(sample.shape[0], device=sample.device) * t
        drift     = to_flattened_numpy(drift_fn(SModel, model, sample, vec_t))
        logp_grad = to_flattened_numpy(div_fn(SModel, model, sample, vec_t, epsilon))
        return np.concatenate([drift, logp_grad], axis=0)

      init       = np.concatenate([to_flattened_numpy(data), np.zeros((shape[0],))], axis=0)
      solution   = integrate.solve_ivp(ode_func, (eps, sde.T), init, rtol=rtol, atol=atol, method=method)
      nfe        = solution.nfev
      zp         = solution.y[:, -1]
      z          = from_flattened_numpy(zp[:-shape[0]], shape).to(data.device).type(torch.float32)
      delta_logp = from_flattened_numpy(zp[-shape[0]:], (shape[0],)).to(data.device).type(torch.float32)
      prior_logp = sde.prior_logp(z)
      bpd        = -(prior_logp + delta_logp) / np.log(2)
      N          = np.prod(shape[1:])
      bpd        = bpd / N
      # A hack to convert log-likelihoods to bits/dim
      offset     = 7. - inverse_scaler(-1.)
      bpd        = bpd + offset
      return bpd, z, nfe

  return likelihood_fn


## Function related to Datasets

In [ ]:

def get_data_scaler(config):
  """
  Data normalizer. Assume data are always in [0, 1].
  """
  if config.data.centered:
    # Rescale to [-1, 1]
    return lambda x: x * 2. - 1.
  else:
    return lambda x: x


def get_data_inverse_scaler(config):
  """
  Inverse data normalizer.
  """
  if config.data.centered:
    # Rescale [-1, 1] to [0, 1]
    return lambda x: (x + 1.) / 2.
  else:
    return lambda x: x

'''
def crop_resize(image, resolution):
  """
  Crop and resize an image to the given resolution.
  """
  crop  = tf.minimum(tf.shape(image)[0], tf.shape(image)[1])
  h, w  = tf.shape(image)[0], tf.shape(image)[1]
  image = image[(h - crop) // 2:(h + crop) // 2,
          (w - crop) // 2:(w + crop) // 2]
  image = tf.image.resize(
    image,
    size=(resolution, resolution),
    antialias=True,
    method=tf.image.ResizeMethod.BICUBIC)
  return tf.cast(image, tf.uint8)


def resize_small(image, resolution):
  """
  Shrink an image to the given resolution.
  """
  h, w  = image.shape[0], image.shape[1]
  ratio = resolution / min(h, w)
  h     = tf.round(h * ratio, tf.int32)
  w     = tf.round(w * ratio, tf.int32)
  return tf.image.resize(image, [h, w], antialias=True)


def central_crop(image, size):
  """
  Crop the center of an image to the given size.
  """
  top  = (image.shape[0] - size) // 2
  left = (image.shape[1] - size) // 2
  return tf.image.crop_to_bounding_box(image, top, left, size, size)


def get_dataset(config, uniform_dequantization=False, evaluation=False):
  """
  Create data loaders for training and evaluation.

  Args:
    config:                 A ml_collection.ConfigDict parsed from config files.
    uniform_dequantization: If `True`, add uniform dequantization to images.
    evaluation:             If `True`, fix number of epochs to 1.

  Returns:
    train_ds, eval_ds, dataset_builder.
  """
  # Compute batch size
  batch_size = config.training.batch_size if not evaluation else config.evaluate.batch_size

  # Reduce this when image resolution is too large and data pointer is stored
  shuffle_buffer_size = 10000
  prefetch_size       = tf.data.experimental.AUTOTUNE
  num_epochs          = None if not evaluation else 1

  # Create dataset builders for each dataset.
  if config.data.dataset == 'CIFAR10':
    dataset_builder  = tfds.builder('cifar10')
    train_split_name = 'train'
    eval_split_name  = 'test'

    def resize_op(img):
      img = tf.image.convert_image_dtype(img, tf.float32)
      return tf.image.resize(img, [config.data.image_size, config.data.image_size], antialias=True)

  elif config.data.dataset == 'SVHN':
    dataset_builder  = tfds.builder('svhn_cropped')
    train_split_name = 'train'
    eval_split_name  = 'test'

    def resize_op(img):
      img = tf.image.convert_image_dtype(img, tf.float32)
      return tf.image.resize(img, [config.data.image_size, config.data.image_size], antialias=True)

  elif config.data.dataset == 'CELEBA':
    dataset_builder  = tfds.builder('celeb_a')
    train_split_name = 'train'
    eval_split_name  = 'validation'

    def resize_op(img):
      img = tf.image.convert_image_dtype(img, tf.float32)
      img = central_crop(img, 140)
      img = resize_small(img, config.data.image_size)
      return img

  elif config.data.dataset == 'LSUN':
    dataset_builder  = tfds.builder(f'lsun/{config.data.category}')
    train_split_name = 'train'
    eval_split_name  = 'validation'

    if config.data.image_size == 128:
      def resize_op(img):
        img = tf.image.convert_image_dtype(img, tf.float32)
        img = resize_small(img, config.data.image_size)
        img = central_crop(img, config.data.image_size)
        return img

    else:
      def resize_op(img):
        img = crop_resize(img, config.data.image_size)
        img = tf.image.convert_image_dtype(img, tf.float32)
        return img

  elif config.data.dataset in ['FFHQ', 'CelebAHQ']:
    dataset_builder  = tf.data.TFRecordDataset(config.data.tfrecords_path)
    train_split_name = eval_split_name = 'train'

  else:
    raise NotImplementedError(
      f'Dataset {config.data.dataset} not yet supported.')

  # Customize preprocess functions for each dataset.
  if config.data.dataset in ['FFHQ', 'CelebAHQ']:
    def preprocess_fn(d):
      sample = tf.io.parse_single_example(d, features={
        'shape': tf.io.FixedLenFeature([3], tf.int64),
        'data': tf.io.FixedLenFeature([], tf.string)})
      data = tf.io.decode_raw(sample['data'], tf.uint8)
      data = tf.reshape(data, sample['shape'])
      data = tf.transpose(data, (1, 2, 0))
      img  = tf.image.convert_image_dtype(data, tf.float32)
      if config.data.random_flip and not evaluation:
        img = tf.image.random_flip_left_right(img)
      if uniform_dequantization:
        img = (tf.random.uniform(img.shape, dtype=tf.float32) + img * 255.) / 256.
      return dict(image=img, label=None)

  else:
    def preprocess_fn(d):
      """
      Basic preprocessing function scales data to [0, 1) and randomly flips.
      """
      img = resize_op(d['image'])
      if config.data.random_flip and not evaluation:
        img = tf.image.random_flip_left_right(img)
      if uniform_dequantization:
        img = (tf.random.uniform(img.shape, dtype=tf.float32) + img * 255.) / 256.

      return dict(image=img, label=d.get('label', None))

  def create_dataset(dataset_builder, split):
    dataset_options = tf.data.Options()
    dataset_options.experimental_optimization.map_parallelization = True
    dataset_options.experimental_threading.private_threadpool_size = 48
    dataset_options.experimental_threading.max_intra_op_parallelism = 1
    read_config = tfds.ReadConfig(options=dataset_options)
    if isinstance(dataset_builder, tfds.core.DatasetBuilder):
      dataset_builder.download_and_prepare()
      ds = dataset_builder.as_dataset(
        split=split, shuffle_files=True, read_config=read_config)
    else:
      ds = dataset_builder.with_options(dataset_options)
    ds = ds.repeat(count=num_epochs)
    ds = ds.shuffle(shuffle_buffer_size)
    ds = ds.map(preprocess_fn, num_parallel_calls=tf.data.experimental.AUTOTUNE)
    ds = ds.batch(batch_size, drop_remainder=True)
    return ds.prefetch(prefetch_size)

  train_ds = create_dataset(dataset_builder, train_split_name)
  eval_ds  = create_dataset(dataset_builder, eval_split_name)
  return train_ds, eval_ds, dataset_builder
'''

## Create a custom Dataset from the images in a folder

In [ ]:
class CustomDataSet(Dataset):

    def __init__(self, root_dir, transform):
        self.root_dir     = root_dir
        self.transform    = transform
        self.all_images   = os.listdir(root_dir)
        self.total_images = natsorted(self.all_images)

    def __len__(self):
        return len(self.total_images)

    def __getitem__(self, idx):
        img_loc      = os.path.join(self.root_dir, self.total_images[idx])
        image        = Image.open(img_loc).convert("RGB")
        tensor_image = self.transform(image)
        return tensor_image

## Create a DataLoader

Define the transformation that will be applied to the images:

* convert the images to tensors
* crop the images
* resize the images
* normalize the images.

Instantiate a Custom Dataset
Create a training DataLoader

In [ ]:
class Data_Loader():
    '''
    DataLoader class that works with LSUN and CelebA datasets.
    '''
    def __init__(
            self,
            dataset,
            images_path,
            image_size,
            crop_size,
            resize,
            normalize,
            centercrop,
            batch_size,
            shuffle = True,
        ):
        self.dataset_name = dataset
        self.path         = images_path
        self.image_size   = image_size
        self.crop_size    = crop_size
        self.resize       = resize
        self.normalize    = normalize
        self.centercrop   = centercrop
        self.batch_size   = batch_size
        self.shuffle      = shuffle
        self.length       = 0 

    def transform(self):

        options = []

        options.append(transforms.ToTensor())
        print('[INFO] Added ToTensor transform ...')

        if self.centercrop:
            offset_height = (218 - self.crop_size) // 2
            offset_width  = (178 - self.crop_size) // 2
            crop = lambda x: x[:, offset_height:offset_height + self.crop_size, offset_width:offset_width + self.crop_size]
            options.append(transforms.Lambda(crop))
            print('[INFO] Added Crop transform ...') 

        if self.resize:
            options.append(transforms.Resize((self.image_size, self.image_size)))
            print('[INFO] Added Resize transform ...') 

        if self.normalize:
            options.append(transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)))
            print('[INFO] Added Normalize transform ...')

        transform = transforms.Compose(options)

        return transform

    def load_lsun(self, classes='church_outdoor_train'):
        transforms = self.transform()
        dataset    = dsets.LSUN(self.path, classes=[classes], transform=transforms)
        return dataset

    def load_celeb(self):
        transforms = self.transform()
        dataset    = CustomDataSet(
            root_dir  = self.path,
            transform = transforms,
        )
        return dataset

    def loader(self):
        if self.dataset_name == 'lsun':
            self.dataset = self.load_lsun()
        elif self.dataset_name == 'CELEBA balanced':
            self.dataset = self.load_celeb()
        size        = len(self.dataset)
        self.length = int(size / self.batch_size)
        print(f'[INFO] Dataset length:    {size}')
        print(f'[INFO] DataLoader length: {self.length}')

        loader = torch.utils.data.DataLoader(
            dataset     = self.dataset,
            batch_size  = self.batch_size,
            shuffle     = self.shuffle,
            num_workers = 2,
            drop_last   = True,
        )
        return loader


In [ ]:
def check_dataloader(
        dataset,
        images_path,
        image_size,
        crop_size,
        resize,
        normalize,
        centercrop,
        batch_size,
        shuffle=True
    ):
    NR, NC    = 3, 3

    DLoader = Data_Loader(
        dataset     = dataset,
        images_path = images_path,
        image_size  = image_size,
        crop_size   = crop_size,
        resize      = resize,
        normalize   = normalize,
        centercrop  = centercrop,
        batch_size  = batch_size,
        shuffle     = shuffle,
    )

    loader = DLoader.loader()
    imgs   = next(iter(loader))

    print(f'[INFO] Batch of images shape: {imgs.shape}')   # BS, Ch, H, W

    if NR*NC > imgs.shape[0]:
        NR = 2
        if NR*NC > imgs.shape[0]:
            NR = 1
            if NR*NC > imgs.shape[0]:
                NC = 2

    _, ax    = plt.subplots(NR, NC, figsize=(3*NC,3*NR))
    plt.suptitle(
        f'Some real images of {dataset} dataset',
        fontsize   = 15,
        fontweight = 'bold',
    )

    index = 0
    for r in range(NR):
        for c in range(NC):
            index += 1
            if NR==1:
                ax[c].imshow(imgs[index].permute(1,2,0))
            else:
                ax[r][c].imshow(imgs[index].permute(1,2,0))

In [ ]:
# Check dataloader

check_dataloader(
    dataset     = config.data.dataset,
    images_path = config.data.data_path,
    image_size  = config.data.image_size,
    crop_size   = config.data.crop_size,
    resize      = config.data.resize,
    normalize   = config.data.normalize,
    centercrop  = config.data.centercrop,
    batch_size  = config.training.batch_size,
)

## Trainer Class

In [ ]:
class Trainer():
    '''
    Class to manage training and evaluation of the model.
    '''
    # =========================================================================
    def __init__(
        self,
        config
        ):
        self.config          = config
        self.score_model     = ScoreModel(config)
        self.SMoptimizer     = None
        self.optimizer       = None
        self.ema             = None
        self.sde             = None
        self.epoch           = 0
        self.step            = 0
        self.train_loader    = None
        self.val_loader      = None

    # =========================================================================
    def get_sde_loss_fn(
        self,
        sde,
        mymodel,
        train,
        reduce_mean          = True,
        continuous           = True,
        likelihood_weighting = True,
        eps                  = 1e-5,
        ):
        """
        Create a loss function for training with arbitrary SDEs.

        Args:
        sde:           An `sde_lib.SDE` object that represents the forward SDE.
        mymodel:       The score model to be trained/evaluated.
        train:        `True` for training loss and `False` for evaluation loss.
        reduce_mean:   If `True`, average the loss across data dimensions. 
                       Otherwise sum the loss across data dimensions.
        continuous:   `True` indicates that the model is defined to take continuous time steps.
                       Otherwise it requires ad-hoc interpolation to take continuous time steps.
        likelihood_weighting: If `True`, weight the mixture of score matching losses
                       according to https://arxiv.org/abs/2101.09258; otherwise use the weighting 
                       recommended in our paper.
        eps:           A `float` number. The smallest time step to sample from.

        Returns:
        A loss function.
        """
        reduce_op = torch.mean if reduce_mean else lambda *args, **kwargs: 0.5 * torch.sum(*args, **kwargs)

        def loss_fn(batch):
            """
            Compute the loss function.

            Args:
                batch: A mini-batch of training data.

            Returns:
                loss: A scalar that represents the average loss value across the mini-batch.
            """
            score_f        = self.score_model.get_score_fn(sde, mymodel, train=train, continuous=continuous)
            t              = torch.rand(batch.shape[0], device=batch.device) * (sde.T - eps) + eps
            z              = torch.randn_like(batch)

            mean, std      = sde.marginal_prob(batch, t)
            perturbed_data = mean + std[:, None, None, None] * z
            score          = score_f(perturbed_data, t)

            if not likelihood_weighting:
                losses = torch.square(score * std[:, None, None, None] + z)
                losses = reduce_op(losses.reshape(losses.shape[0], -1), dim=-1)
            else:
                g2     = sde.sde(torch.zeros_like(batch), t)[1] ** 2
                losses = torch.square(score + z / std[:, None, None, None])
                losses = reduce_op(losses.reshape(losses.shape[0], -1), dim=-1) * g2

            loss = torch.mean(losses)
            return loss

        return loss_fn

    # =========================================================================
    def get_smld_loss_fn(self, vesde, mymodel, train, reduce_mean=False):
        """
        Legacy code to reproduce previous results on SMLD(NCSN). Not recommended for new work.
        """
        assert isinstance(vesde, VESDE), "SMLD training only works for VESDEs."

        # Previous SMLD models assume descending sigmas
        smld_sigma_array = torch.flip(vesde.discrete_sigmas, dims=(0,))
        reduce_op        = torch.mean if reduce_mean else lambda *args, **kwargs: 0.5 * torch.sum(*args, **kwargs)

        def loss_fn(batch):
            model_fn       = self.score_model.get_model_fn(mymodel, train=train)
            labels         = torch.randint(0, vesde.N, (batch.shape[0],), device=batch.device)
            sigmas         = smld_sigma_array.to(batch.device)[labels]
            noise          = torch.randn_like(batch) * sigmas[:, None, None, None]
            perturbed_data = noise + batch
            score          = model_fn(perturbed_data, labels)

            target         = -noise / (sigmas ** 2)[:, None, None, None]
            losses         = torch.square(score - target)
            losses         = reduce_op(losses.reshape(losses.shape[0], -1), dim=-1) * sigmas ** 2
            loss           = torch.mean(losses)
            return loss

        return loss_fn

    # =========================================================================
    def get_ddpm_loss_fn(self, vpsde, mymodel, train, reduce_mean=True):
        """
        Legacy code to reproduce previous results on DDPM. Not recommended for new work.
        """
        assert isinstance(vpsde, VPSDE), "DDPM training only works for VPSDEs."

        reduce_op = torch.mean if reduce_mean else lambda *args, **kwargs: 0.5 * torch.sum(*args, **kwargs)

        def loss_fn(batch):
            model_fn               = self.score_model.get_model_fn(mymodel, train=train)
            labels                 = torch.randint(0, vpsde.N, (batch.shape[0],), device=batch.device)
            sqrt_alphas_cumprod    = vpsde.sqrt_alphas_cumprod.to(batch.device)
            sqrt_1m_alphas_cumprod = vpsde.sqrt_1m_alphas_cumprod.to(batch.device)
            noise                  = torch.randn_like(batch)
            perturbed_data         = sqrt_alphas_cumprod[labels, None, None, None] * batch + \
                                     sqrt_1m_alphas_cumprod[labels, None, None, None] * noise
            score                  = model_fn(perturbed_data, labels)

            losses                 = torch.square(score - noise)
            losses                 = reduce_op(losses.reshape(losses.shape[0], -1), dim=-1)
            loss                   = torch.mean(losses)
            return loss

        return loss_fn

    # =========================================================================
    def get_step_fn(
        self,
        sde,
        mymodel,
        train,
        optimize_fn          = None,
        reduce_mean          = False,
        continuous           = True,
        likelihood_weighting = False,
        ):
        """
        Creates a function for one-step training/evaluation.

        Args:
            sde:          An `sde_lib.SDE` object that represents the forward SDE.
            mymodel:      The score model to be trained/evaluated.
            train:       `True` for training loss and `False` for evaluation loss.
            optimize_fn:  An optimization function.
            reduce_mean:  If `True`, average the loss across data dimensions. Otherwise sum the loss across data dimensions.
            continuous:  `True` indicates that the model is defined to take continuous time steps.
            likelihood_weighting: If `True`, weight the mixture of score matching losses according to
            https://arxiv.org/abs/2101.09258; otherwise use the weighting recommended by our paper.

        Returns:
            A function for one-step training or evaluation.
        """
        if continuous:
            loss_fn = self.get_sde_loss_fn(
                sde,
                mymodel,
                train,
                reduce_mean          = reduce_mean,
                continuous           = True,
                likelihood_weighting = likelihood_weighting,
            )
        else:
            assert not likelihood_weighting, "Likelihood weighting is not supported for original SMLD/DDPM training."
            if isinstance(sde, VESDE):
                loss_fn = self.get_smld_loss_fn(sde, mymodel, train, reduce_mean=reduce_mean)
            elif isinstance(sde, VPSDE):
                loss_fn = self.get_ddpm_loss_fn(sde, mymodel, train, reduce_mean=reduce_mean)
            else:
                raise ValueError(f"Discrete training for {sde.__class__.__name__} is not recommended.")

        def step_fn(batch):
            """
            Running one step of training or evaluation.

            This function will undergo `jax.lax.scan` so that multiple steps can be 
            mapped and jit-compiled together for faster execution.

            Args:
                batch: A mini-batch of training/evaluation data.

            Returns:
                loss: The average loss value of this state.
            """
            if train:
                # Reset gradients only after 'config.training.batches_accum_gradients' iterations
                if self.step % self.config.training.batches_accum_gradients == 0:
                    self.optimizer.zero_grad()
                loss = loss_fn(batch)
                loss.backward()
                # Reset gradients only after 'config.training.batches_accum_gradients' iterations
                if (self.step+1) % self.config.training.batches_accum_gradients == 0:
                    optimize_fn(self.optimizer, self.score_model.model, step=self.step)
                    self.ema.update(self.score_model.model.parameters())
            else:
                with torch.no_grad():
                    self.ema.store(self.score_model.model)
                    self.ema.copy_to(self.score_model.model)
                    loss = loss_fn(batch)
                    self.ema.restore(self.score_model.model)

            return loss

        return step_fn

    # ==================================================================================
    def train(self, results):
        '''
        Train the model.
        '''
        # Initialize the model EMA, optimizer, parameters ...............................

        self.ema = ExponentialMovingAverage(
            self.score_model.model,
            decay = self.config.model.ema_rate,
        )

        # Class that supports the optimizer
        self.SMoptimizer = ScoreModelOptimizer(
            self.config,
            self.score_model.model,
        )
        # Link to the pytorch optimizer of the class 'self.SMoptimizer'
        self.optimizer = self.SMoptimizer.optimizer

        self.step      = 0 # Initialize the step counter

        # Continue previous training when a checkpoint is detected ......................

        self.restore_checkpoint()
        initial_step = int(self.step)

        # Create the training data loader ...............................................

        TDLoader = Data_Loader(
            dataset     = config.data.dataset,
            images_path = config.data.data_path,
            image_size  = config.data.image_size,
            crop_size   = config.data.crop_size,
            resize      = config.data.resize,
            normalize   = config.data.normalize,
            centercrop  = config.data.centercrop,
            batch_size  = config.training.batch_size,
            shuffle     = True,
        )
        self.train_loader = TDLoader.loader()

        # Setup dataset dependent hyperparameters .......................................

        dataset_len                    = TDLoader.length
        config.training.n_iters        = int(config.training.epochs * dataset_len)
        config.training.snapshot_freq  = int(config.training.snapshot_freq * dataset_len)

        print(f'[INFO] Dataset length: {dataset_len} batches of {config.training.batch_size} images each')
        print(f'[INFO] epoch: {self.epoch} :: step: {self.step} :: snapshot freq: {config.training.snapshot_freq}')

        # Create the validation data loader .............................................

        VDLoader = Data_Loader(
            dataset     = config.data.dataset,
            images_path = config.evaluate.data_path,
            image_size  = config.data.image_size,
            crop_size   = config.data.crop_size,
            resize      = config.data.resize,
            normalize   = config.data.normalize,
            centercrop  = config.data.centercrop,
            batch_size  = config.training.batch_size,
            shuffle     = False,
        )
        self.val_loader = VDLoader.loader()

        val_dataset_len = VDLoader.length
        print(f'[INFO] Validation dataset length: {val_dataset_len} batches of {config.evaluate.batch_size} images each')

        # Create data normalizer and its inverse ........................................

        scaler         = get_data_scaler(self.config)
        inverse_scaler = get_data_inverse_scaler(self.config)

        # Setup the SDE .................................................................

        if self.config.training.sde.lower() == 'vpsde':
            print(f'[INFO] creating VPSDE ...')
            self.sde = VPSDE(
                beta_min = self.config.model.beta_min, 
                beta_max = self.config.model.beta_max, 
                N        = self.config.model.num_scales,
            )
            sampling_eps = 1e-3

        elif self.config.training.sde.lower() == 'subvpsde':
            print(f'[INFO] creating SUBVPSDE ...')
            self.sde = subVPSDE(
                beta_min = self.config.model.beta_min,
                beta_max = self.config.model.beta_max,
                N        = self.config.model.num_scales,
            )
            sampling_eps = 1e-3

        elif self.config.training.sde.lower() == 'vesde':
            print(f'[INFO] creating VESDE ...')
            self.sde = VESDE(
                sigma_min = self.config.model.sigma_min,
                sigma_max = self.config.model.sigma_max,
                N         = self.config.model.num_scales,
            )
            sampling_eps = 1e-5

        else:
            raise NotImplementedError(f"SDE {self.config.training.sde} unknown.")

        # Define the functions for one-step training and evaluation ....................

        optimize_fn          = self.SMoptimizer.optimization_manager(self.config)
        continuous           = self.config.training.continuous
        reduce_mean          = self.config.training.reduce_mean
        likelihood_weighting = self.config.training.likelihood_weighting

        train_step_fn = self.get_step_fn(
            self.sde,
            mymodel              = self.score_model.model,
            train                = True,
            optimize_fn          = optimize_fn,
            reduce_mean          = reduce_mean,
            continuous           = continuous,
            likelihood_weighting = likelihood_weighting,
        )
        eval_step_fn = self.get_step_fn(
            self.sde,
            mymodel              = self.score_model.model,
            train                = False,
            optimize_fn          = optimize_fn,
            reduce_mean          = reduce_mean,
            continuous           = continuous,
            likelihood_weighting = likelihood_weighting,
        )

        # Define the sampling function..................................................

        if self.config.training.snapshot_sampling:
            sampling_shape = (
                self.config.training.batch_size,
                self.config.data.num_channels,
                self.config.data.image_size,
                self.config.data.image_size
            )
            sampling_fn = get_sampling_fn(
                self.score_model,
                self.score_model.model,
                self.config,
                self.sde,
                sampling_shape,
                inverse_scaler,
                sampling_eps,
            )

        num_train_steps = self.config.training.n_iters
        accum_loss      = 0.0

        print(f"[INFO] Starting training loop at epoch {self.epoch} and step {initial_step}")

        # Iterate over all the training steps ..........................................

        for epoch in range(self.epoch, config.training.epochs):

            ts  = time.time()

            loop = tqdm(self.train_loader, leave=True)

            for batch_id, batch in enumerate(loop):

                # Reset accumulated loss afer 'config.training.batches_accum_gradients' iterations

                if self.step % self.config.training.batches_accum_gradients == 0:
                    accum_loss = 0.0

                batch = batch.to(self.config.device)
                batch = scaler(batch)

                # Execute one training step .................................................

                loss = train_step_fn(batch)

                # Accumulate the loss during 'config.training.batches_accum_gradients' iterations
                # and compute & print the averaged loss after that number of iterations

                accum_loss += loss.item()
                if (self.step+1) % self.config.training.batches_accum_gradients == 0:
                    accum_loss /= self.config.training.batches_accum_gradients
                    results["loss"].append(accum_loss)

                # Print and log the training and validation losses periodically .............

                log_training = ((self.step+1) % self.config.training.log_freq  == 0)
                log_eval     = ((self.step+1) % self.config.training.eval_freq == 0)

                if log_training==True and log_eval==False:

                    print(f'epoch: {self.epoch+1} | step: {self.step+1} | training_loss: {accum_loss :.5e}')
                    mean_loss = np.mean(results["loss"][-self.config.training.log_freq:])

                    try:
                        # Log metrics to Weights and Biases .........................
                        wandb.log(
                            {
                            "loss":    mean_loss,
                            }
                        )
                    except Exception as e:
                        print(f'[ERROR] (#1) An exception of type {type(e).__name__} occurred. Arguments:\n{ex.args!r}')

                elif log_training==True and log_eval==True:

                    mean_loss = np.mean(results["loss"][-self.config.training.log_freq:])

                    val_batch = next(iter(self.val_loader)).to(self.config.device)
                    val_batch = scaler(val_batch)
                    val_loss  = eval_step_fn(val_batch)
                    print(f'epoch: {self.epoch+1} | step: {self.step+1} | training_loss: {accum_loss :.5e} | val_loss: {val_loss.item(): .5e}')

                    try:
                        # Log metrics to Weights and Biases .........................
                        wandb.log(
                            {
                            "loss":     mean_loss,
                            "val_loss": val_loss.item(),
                            }
                        )
                    except Exception as ex:
                        print(f'[ERROR] (#2) An exception of type {type(ex).__name__} occurred. Arguments:\n{ex.args!r}')

                elif log_training==False and log_eval==True:

                    val_batch = next(iter(self.val_loader)).to(self.config.device)
                    val_batch = scaler(val_batch)
                    val_loss  = eval_step_fn(val_batch)
                    print(f'epoch: {self.epoch+1} | step: {self.step+1} | val_loss: {val_loss.item(): .5e}')

                    try:
                        # Log metrics to Weights and Biases .........................
                        wandb.log(
                            {
                            "val_loss": val_loss.item(),
                            }
                        )
                    except Exception as ex:
                        print(f'[ERROR] (#3) An exception of type {type(ex).__name__} occurred. Arguments:\n{ex.args!r}')


                # Save a checkpoint periodically and generate samples when configured to do so

                if self.step != 0 and (self.step+1) % config.training.snapshot_freq == 0 or (self.step+1) == num_train_steps:

                    # Save the checkpoint .......................................................

                    model_file = self.set_file_name(
                        config,
                        config.experiment.models_dir,
                        postfix=None,
                        extension='pth',
                    )
                    self.save_checkpoint(model_file)

                    # Generate and save samples ...............................................

                    if self.config.training.snapshot_sampling:
                        self.ema.store(self.score_model.model)
                        self.ema.copy_to(self.score_model.model)
                        sample, n  = sampling_fn(self.score_model, self.score_model.model)

                        self.ema.restore(self.score_model.model)
                        nrow       = int(np.sqrt(sample.shape[0]))
                        image_grid = make_grid(sample, nrow, padding=2)
                        #image_npy = np.clip(sample.cpu().numpy() * 255, 0, 255).astype(np.uint8)

                        # Save the grid of images as a PNG file

                        file_png = self.set_file_name(
                            config,
                            config.experiment.results_dir,
                            postfix   = '_generated',
                            extension = 'png',
                        )
                        save_image(image_grid, file_png)

                # Compute evaluation metrics ............................................

                if self.config.evaluate.calc_metrics_interval > 0 and (self.step+1) % self.config.evaluate.calc_metrics_interval == 0 and self.step != 0:

                    try:
                        metrics_dic = self.calculate_metrics(sampling_eps)
                        self.last_metrics = metrics_dic
                    except ValueError as e:
                        print(f"[ERROR] exception {e} in calculate_metrics()")

                    metrics_path = os.path.join(
                        config.experiment.root_dir,
                        config.experiment.results_dir,
                        config.experiment.experiment_name,
                        f'metrics_scores.txt'
                    )

                    with open(metrics_path, 'a') as f: 
                        f.write(f"{self.step+1} , {self.last_metrics['inception_score_mean']} , \
                            {self.last_metrics['inception_score_std']} , \
                            {self.last_metrics['frechet_inception_distance']} , \
                            {self.last_metrics['kernel_inception_distance_mean']} , \
                            {self.last_metrics['kernel_inception_distance_std']}\n")

                self.step += 1

            # End of an epoch .....................................................

            te        = time.time()
            texec_sec = te - ts
            texec_str = time_format(texec_sec)
            print(f'\nEpoch {self.epoch+1} training time: {texec_str}')
            results['epoch_training_time'].append(texec_sec)

            try:
                wandb.log(
                    {
                    "epoch_training_time_sec": texec_sec,
                    }
                )
            except Exception as ex:
                print(f'[ERROR] (#4) An exception of type {type(ex).__name__} occurred. Arguments:\n{ex.args!r}')

            self.epoch += 1


    # ========================================================================================
    def evaluate(
        self,
        results,
        ):
        """
        Evaluate the trained model.
        """

        # Create data loader for likelihood evaluation. ......................................
        # # Only evaluate on uniformly dequantized data

        if self.config.evaluate.bpd_dataset.lower() == 'train':

            VDLoader = Data_Loader(
                dataset     = config.data.dataset,
                images_path = config.data.data_path,
                image_size  = config.data.image_size,
                crop_size   = config.data.crop_size,
                resize      = config.data.resize,
                normalize   = config.data.normalize,
                centercrop  = config.data.centercrop,
                batch_size  = config.evaluate.batch_size,
                shuffle     = True,
            )
            self.val_loader = VDLoader.loader()
            bpd_num_repeats = 1

        elif self.config.evaluate.bpd_dataset.lower() == 'test':

            VDLoader = Data_Loader(
                dataset     = config.data.dataset,
                images_path = config.evaluate.data_path,
                image_size  = config.data.image_size,
                crop_size   = config.data.crop_size,
                resize      = config.data.resize,
                normalize   = config.data.normalize,
                centercrop  = config.data.centercrop,
                batch_size  = config.evaluate.batch_size,
                shuffle     = False,
            )
            self.val_loader = VDLoader.loader()
            bpd_num_repeats = 5

        else:
            raise ValueError(f"[ERROR] Configuration option evaluate.bpd_dataset={self.config.evaluate.bpd_dataset} invalid.")

        val_dataset_len = VDLoader.length
        print(f'[INFO] Validation dataset length: {val_dataset_len} batches of {config.evaluate.batch_size} images each')

        # Create data normalizer and its inverse .....................................

        scaler         = get_data_scaler(self.config)
        inverse_scaler = get_data_inverse_scaler(self.config)

        self.ema = ExponentialMovingAverage(
            self.score_model.model,
            decay = self.config.model.ema_rate,
        )

        # Class that supports the optimizer
        self.SMoptimizer = ScoreModelOptimizer(
            self.config,
            self.score_model.model,
        )
        # Link to the pytorch optimizer of the class 'self.SMoptimizer'
        self.optimizer = self.SMoptimizer.optimizer

        self.step      = 0 # Initialize the step counter

        # Setup SDEs .................................................................

        if self.config.training.sde.lower() == 'vpsde':
            self.sde     = VPSDE(
                beta_min = self.config.model.beta_min,
                beta_max = self.config.model.beta_max,
                N        = self.config.model.num_scales,
            )
            sampling_eps = 1e-3
        elif self.config.training.sde.lower() == 'subvpsde':
            self.sde     = subVPSDE(
                beta_min = self.config.model.beta_min,
                beta_max = self.config.model.beta_max, 
                N        = self.config.model.num_scales
            )
            sampling_eps = 1e-3
        elif self.config.training.sde.lower() == 'vesde':
            self.sde     = VESDE(
                sigma_min = self.config.model.sigma_min,
                sigma_max = self.config.model.sigma_max,
                N         = self.config.model.num_scales
            )
            sampling_eps = 1e-5
        else:
            raise NotImplementedError(f"SDE {self.config.training.sde} unknown.")

        # Create the one-step evaluation function when loss computation is enabled ....

        if self.config.evaluate.enable_loss:
            optimize_fn          = self.SMoptimizer.optimization_manager(self.config)
            continuous           = self.config.training.continuous
            likelihood_weighting = self.config.training.likelihood_weighting

            reduce_mean = self.config.training.reduce_mean
            eval_step = self.get_step_fn(
                self.sde,
                mymodel              = self.score_model.model,
                train                = False,
                optimize_fn          = optimize_fn,
                reduce_mean          = reduce_mean,
                continuous           = continuous,
                likelihood_weighting = likelihood_weighting,
            )

        # Build the likelihood computation function when it is enabled .......
        if self.config.evaluate.enable_bpd:
            likelihood_fn = get_likelihood_fn(
                self.score_model,
                self.score_model.model,
                self.sde,
                inverse_scaler,
            )

        # Build the sampling function when sampling is enabled .......................
        if self.config.evaluate.enable_sampling:
            sampling_shape = (
                self.config.evaluate.batch_size,
                self.config.data.num_channels,
                self.config.data.image_size,
                self.config.data.image_size
                )
            sampling_fn = get_sampling_fn(
                self.score_model,
                self.score_model.model,
                self.config,
                self.sde,
                sampling_shape,
                inverse_scaler,
                sampling_eps,
                )

        # Restore the specified saved checkpoint ......................................

        self.restore_checkpoint()

        self.ema.copy_to(self.score_model.model)

        # Compute the loss on the full evaluation dataset if loss computation is enabled

        if self.config.evaluate.enable_loss:

            all_losses = []
            loop       = tqdm(self.val_loader, leave=True)

            for i, batch in enumerate(loop):

                val_batch = batch.to(self.config.device)
                val_batch = scaler(val_batch)
                val_loss  = eval_step(val_batch)
                all_losses.append(val_loss.item())
                if (i + 1) % 1000 == 0:
                    print(f'[INFO] Finished {i+1}-th step of loss evaluation')

            # Save loss values to file
            all_losses = np.asarray(all_losses)
            filename   = self.set_file_name(
                config,
                config.experiment.results_dir,
                postfix   = '_valLoss',
                extension = 'npz',
            )
            np.savez_compressed(filename, all_losses)

        # Compute log-likelihoods (bits/dim) if enabled .................................

        if self.config.evaluate.enable_bpd:

            bpds = []
            for repeat in range(bpd_num_repeats):

                loop = tqdm(self.val_loader, leave=True)

                for i, batch in enumerate(loop):

                    val_batch  = batch.to(self.config.device)
                    val_batch  = scaler(val_batch)
                    bpd        = likelihood_fn(self.score_model, self.score_model.model, val_batch)[0]
                    bpd        = bpd.detach().cpu().numpy().reshape(-1)
                    bpds.extend(bpd)

                    print(f"Log likelihood > repeat: {repeat} | batch: {i+1} | mean bpd: {np.mean(np.asarray(bpds)) :.6f}")

            # Save bits/dim to disk
            bpds = np.asarray(bpds)
            filename   = self.set_file_name(
                config,
                config.experiment.results_dir,
                postfix='_bitsPerDim',
                extension='npz',
            )
            np.savez_compressed(filename, bpds)

        # Calculate and save metrics ....................................................

        metrics_path = os.path.join(
            config.experiment.root_dir,
            config.experiment.results_dir,
            config.experiment.experiment_name,
            f'metrics_scores.txt'
        )
        with open(metrics_path, 'a') as f:
            f.write(f'IS_mean , IS_sdev, FID , KID_mean , KID_sdev\n')

        self.last_metrics = None
        try:
            metrics_dic       = self.calculate_metrics(sampling_eps)
            self.last_metrics = metrics_dic
        except ValueError as e:
            print("[ERROR] exception {e} in calculate_metrics()")

        if self.last_metrics is not None:
            with open(metrics_path, 'a') as f: 
                f.write(f"{self.step+1} , {self.last_metrics['inception_score_mean']} , \
                    {self.last_metrics['inception_score_std']} , \
                    {self.last_metrics['frechet_inception_distance']} , \
                    {self.last_metrics['kernel_inception_distance_mean']} , \
                    {self.last_metrics['kernel_inception_distance_std']}\n")


    # ==================================================================
    def restore_checkpoint(self):
        '''
        Restore a model checkpoint from file.
        '''
        if self.config.experiment.saved_model is not None:
            model_file  = os.path.join(
                self.config.experiment.root_dir,
                self.config.experiment.models_dir,
                self.config.experiment.experiment_name,
                self.config.experiment.saved_model
            )
            if os.path.isfile(model_file) == True:
                loaded_state = torch.load(model_file, weights_only=False, map_location=self.config.device)
                self.optimizer.load_state_dict(loaded_state['optimizer'])
                self.SMoptimizer.optimizer = self.optimizer
                self.score_model.model.load_state_dict(loaded_state['model'], strict=False)
                self.ema.load_state_dict(loaded_state['ema'])
                self.epoch = loaded_state['epoch'] + 1
                self.step  = loaded_state['step'] + 1
                print(f'[INFO] Loaded checkpoint from {model_file}!')
                return
            else:
                print(f'[ERROR] Checkpoint {model_file} does not exist!')

    # ==================================================================
    def save_checkpoint(self, model_file):
        '''
        Save a model checkpoint to file.
        '''
        self.SMoptimizer.optimizer = self.optimizer 
        saved_state = {
            'optimizer': self.optimizer.state_dict(),
            'model':     self.score_model.model.state_dict(),
            'ema':       self.ema.state_dict(),
            'epoch':     self.epoch,
            'step':      self.step
        }
        torch.save(saved_state, model_file)
        print(f'[INFO] Saved checkpoint {model_file}!')

    # ==================================================================
    def set_file_name(self, config, middle, postfix=None, extension='pth'):
        '''
        Compose a file path given current configuration, epoch, step, postfix and extension.
        '''
        fname_base = f'{config.experiment.experiment_name}_epoch{str(self.epoch+1).zfill(3)}_step{str(self.step+1).zfill(8)}'
        if postfix is not None:
            fname_last = f'{fname_base}{postfix}.{extension}'
        else:
            fname_last = f'{fname_base}.{extension}'

        fname = os.path.join(
            config.experiment.root_dir,
            middle,
            config.experiment.experiment_name,
            fname_last
        )
        return fname

    # =================================================================
    @torch.no_grad()
    def calculate_metrics(self, sampling_eps):
        '''
        Calculates the FID and KID between 'num_batches' batches of real images and generated images.
        Calculates the IS in 'num_batches' batches of generated images.

        Returns:
        	Dictionary 
			{
            'inception_score_mean':           valorISm,
            'inception_score_std':            valorISd,
            'frechet_inception_distance':     valorFID,
            'kernel_inception_distance_mean': valorKIDm,
            'kernel_inception_distance_std':  valrKIDd
			}
        '''
        torch.cuda.empty_cache()

        real_path = os.path.join(
        	self.config.experiment.root_dir,
        	self.config.experiment.results_dir,
            self.config.experiment.experiment_name,
        	self.config.evaluate.metrics_dir,
        	'real'
        )
        fake_path = os.path.join(
        	self.config.experiment.root_dir,
        	self.config.experiment.results_dir,
            self.config.experiment.experiment_name,
        	self.config.evaluate.metrics_dir,
        	'fake'
        )

        scalerM         = get_data_scaler(self.config)
        inverse_scalerM = get_data_inverse_scaler(self.config)

        sampling_shapeM = (
            self.config.evaluate.calc_metrics_batch_size,
            self.config.data.num_channels,
            self.config.data.image_size,
            self.config.data.image_size,
        )
        sampling_fn = get_sampling_fn(
            self.score_model,
            self.score_model.model,
            self.config,
            self.sde,
            sampling_shapeM,
            inverse_scalerM,
            sampling_eps,
        )

        num_batches = int(np.ceil(self.config.evaluate.num_samples / self.config.evaluate.calc_metrics_batch_size))

        # Remove any existing real images used for metrics calculation.
        # Recreate directories used for metrics calculation.
        # Copy real images that will be used in metrics calculation.

        if os.path.exists(real_path) and self.config.evaluate.clear_metrics_cache:
            rmtree(real_path, ignore_errors=True)

        if os.path.exists(real_path) == False:

            print('[INFO] calculating metrics - copy real images')

            os.makedirs(real_path)

            print(f'[INFO] real images source path: {self.config.evaluate.calc_metrics_path}')
            files = glob.glob(self.config.evaluate.calc_metrics_path+'/*.jpg')
            print(f'[INFO] number of real images in source path: {len(files)}')

            if (self.config.evaluate.num_samples % self.config.evaluate.calc_metrics_batch_size != 0):
                quoc = self.config.evaluate.num_samples // self.config.evaluate.calc_metrics_batch_size
                self.config.evaluate.num_samples = (quoc + 1) * self.config.evaluate.calc_metrics_batch_size

            sample = random.sample(files, k=self.config.evaluate.num_samples)

            for f,file in enumerate(sample):
                copy(file, real_path)

        else:
            print('[WARN] calculating metrics - using previously copied real images')

        # Generate fake images and place them in folder 'metrics/exp_name/fake'

        rmtree(fake_path, ignore_errors=True)
        os.makedirs(fake_path)

        self.score_model.eval()

        for batch_id in tqdm(range(num_batches), desc='[INFO] calculating metrics - generate fake images'):

            generated_images, nfe = sampling_fn(self.score_model, self.score_model.model)

            # Save the generated images to files
            for j, image in enumerate(generated_images.unbind(0)):
                save_image(
                    image,
                    f'{fake_path}/{j+batch_id * self.config.evaluate.calc_metrics_batch_size}.png',
                )

        # Calculate the metrics using the real and generated images

        metrics_dict = None

        try:
            metrics_dict = torch_fidelity.calculate_metrics(
		        input1          = fake_path,
		        input2          = real_path,
		        cuda            = True,
		        isc             = True,
		        fid             = True,
		        kid             = True,
		        kid_subset_size = self.config.evaluate.kid_subset_size,
		        batch_size      = self.config.evaluate.calc_metrics_batch_size,
		        verbose         = False,
		        feature_extractor_internal_dtype = 'float32',
		        feature_extractor       = "inception-v3-compat",
		        samples_resize_and_crop = self.config.evaluate.samples_resize_crop,
		        feature_layer_fid       = self.config.evaluate.feature_layer_fid,
		    )
        except AssertionError as e:
            print(f'[ERROR] Exception {e} in calculate_metrics()')

        print(f"IS mean:  {metrics_dict['inception_score_mean']} sttdev: {metrics_dict['inception_score_std']}")
        print(f"FID:      {metrics_dict['frechet_inception_distance']}")
        print(f"KID mean: {metrics_dict['kernel_inception_distance_mean']} stddev: {metrics_dict['kernel_inception_distance_std']}")

        self.score_model.model.train()

        return metrics_dict


    def print_model_summary(self, model):
        '''
        Print a summary of the architecture of the NCSN++ model.
        '''

        # _x:      torch.Size([BS, 3, 128, 128])
        # t:       torch.Size([BS])
        # Output:  torch.Size([BS, 3, 128, 128])

        aux_x = torch.randn(
            (
            self.config.training.batch_size,
            self.config.data.num_channels,
            self.config.data.image_size,
            self.config.data.image_size
            )).to(self.config.device)

        aux_t = torch.randn(self.config.training.batch_size).to(self.config.device)
        sumSM = summary(
            model,
            input_data   = [aux_x, aux_t],
            col_width    = 16,
            col_names    = ["kernel_size", "output_size", "num_params"],
            row_settings = ["var_names"],
        )
        print(sumSM)
        del aux_x
        del aux_t


    def save_model_onnx(self, model):
        '''
        Export the score model to ONNX format.
        '''
        file_save_onnx = f'{MODELS_PATH}/{self.config.experiment.experiment_name}_architecture.onnx'

        print("Creating a ONNX file with the model architecture ...")

        model.eval()

        # Export the model

        aux_x = torch.randn(
            (
            self.config.training.batch_size,
            self.config.data.num_channels,
            self.config.data.image_size,
            self.config.data.image_size
            )).to(self.config.device)
        aux_t        = torch.randn(self.config.training.batch_size).to(self.config.device)
        aux_data     = (aux_x, aux_t)
        input_names  = [ "X", "t" ]
        output_names = [ "Out" ]

        torch.onnx.dynamo_export(
            model,
            *aux_data,
        ).save(file_save_onnx)

        # torch.onnx.export(
        #    model,
        #    aux_data,
        #    file_save_onnx,
        #    verbose      = True,
        #    input_names  = input_names,
        #    output_names = output_names,
        # )

        model.train()


In [ ]:
# Create an empty dictionary to store the training results

results = {
    'loss':                [],
    'val_loss':            [],
    'epoch_training_time': [],
}

In [ ]:
def main(results):

  trainer = Trainer(config)

  if config.experiment.mode == "train":
    # Train the model
    trainer.train(results)

  elif config.experiment.mode == "evaluate":
    # Evaluate the model
    trainer.evaluate(results)

  elif config.experiment.mode == "summary":
    # Print a summary of the model architecture
    trainer.print_model_summary(trainer.score_model.model)
    # Save the model architecture to ONNX format
    trainer.save_model_onnx(trainer.score_model.model)

  else:
    raise ValueError(f"Mode {config.mode} not recognized.")


## Login into Weights & Bias

In [ ]:
wandb.login()

## Track metadata and hyperparameters with Weights & Bias

Define the experiment: the hyperparameters, the dataset and model name. This information will be stored in a `config` dictionary.

In [ ]:
config_wandb = config

wandb.init(
    project = 'OUR_WANDB_PROJECT_ID',
    entity  = 'OUR_WANDB_ENTITY', 
    config  = config_wandb,
    #id     = 'OUR_WANDB_RUN_ID', # config.experiment.experiment_name,
    #resume = 'allow'
)

In [ ]:
# Write an header to the metrics file

if config.evaluate.calc_metrics_interval > 0:
    metrics_path = os.path.join(
        config.experiment.root_dir,
        config.experiment.results_dir,
        config.experiment.experiment_name,
        f'metrics_scores.txt'
    )
    with open(metrics_path, 'a') as f:
        f.write(f'IS_mean , IS_sdev, FID , KID_mean , KID_sdev\n')

## Training the Model

In [ ]:
main(results)

## Finishing the Connection to Weights & Bias

In [ ]:
# Mark the Weights and Bias run as finished
wandb.finish()